In [6227]:
gen_report = False
show_plots = False

In [6228]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if "dir" in k.lower() or "split" in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace("experiments/", "").replace("/results", "")
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map(
    {True: "yes", False: "no"}
)
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map(
    {True: "yes", False: "no"}
)

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == "frozen", "learning_rate_phase2_a"] = "-"
df.loc[df["best_of_n"].isna(), "best_of_n"] = 1
df.loc[df["dataset_tt"].isna(), "dataset_tt"] = "one_shape"
df.loc[df["sampling_temperature_tt"].isna(), "sampling_temperature_tt"] = df["sampling_temperature"]
df.loc[df["gumbel"].isna(), "gumbel"] = False

df.loc[(df["num_iterations"] == 0) | (df["best_of_n"] == 1), "test_time_mode"] = "-"

df.rename(
    columns={"test_time_training_accuracy": "test_time_mutual_accuracy"}, inplace=True
)

df.rename(
    columns={"test_time_self_play_accuracy": "test_time_self_accuracy"}, inplace=True
)

df["test_time_mode"] = df["test_time_mode"].replace("adaptation", "sample_adaptation")
df["dataset"] = df["dataset"].replace("shape", "shape1")
df["dataset_tt"] = df["dataset_tt"].replace("two_shape", "shape2")
df["dataset_tt"] = df["dataset_tt"].replace(
    "shape_unique_double_attribute", "dual_attribute_shape"
)
# df["dataset_tt"] = df["dataset_tt"].replace("shape_unique_double_attribute", "single_attribute_shape")

df.loc[df["baseline"].isna(), "baseline"] = False

df.loc[
    df["baseline"]
    & (df["test_time_mode"] != "oracle_adaptation")
    & (df["test_time_mode"] != "oracle_full_adaptation"),
    "test_time_mode",
] = "-"

df["test_time_mutual_accuracy"] = df["test_time_mutual_accuracy"] * 100
df["test_time_self_accuracy"] = df["test_time_self_accuracy"] * 100
df["mutual_play_accuracy"] = df["mutual_play_accuracy"] * 100
df["self_play_accuracy_a"] = df["self_play_accuracy_a"] * 100

df = df.where(pd.notna(df), "None")
pd.options.display.float_format = "{:.10g}".format
df = df.map(
    lambda x: (
        f"{x:.0e}"
        if isinstance(x, (int, float))
        and x != 0
        and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01)
        else x
    )
)
print("Total reports loaded:", len(df))

pd.set_option("display.max_rows", None)

/tmp/ipykernel_3321/99214765.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
/tmp/ipykernel_3321/99214765.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["agent_a_training_mode"] == "frozen", "learning_rate_phase2_a"] = "-"


Total reports loaded: 8993


In [6229]:
def filter_df(
    filters,
    df=df,
    sort_by=[
        "message_length",
        "message_length_tt",
        "learning_rate_tt",
        "num_iterations",
    ],
):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if (type(value) == str) and value.startswith(">"):
                mask |= df[key] > value.split(">")[1]
            if value == "!None":
                mask |= df[key] != "None"
            else:
                mask |= df[key] == value
        idx &= mask

    return df[idx].sort_values(by=sort_by).reset_index(drop=True)

In [6230]:
import os
import shutil


def clean_expr(res_df, check_before_delete=True):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        if check_before_delete:
            answer = (
                input(f"⚠️ Remove all files in '{path}' except report.json? (y/n): ")
                .strip()
                .lower()
            )
        else:
            answer = "y"

        if answer != "y":
            print(f"❌ Skipped: {path}")
            continue

        for item in os.listdir(full_path):
            print(item)
            if item == "results":
                continue

            item_path = os.path.join(full_path, item)

            try:
                if os.path.isdir(item_path):
                    shutil.rmtree(item_path)
                else:
                    os.remove(item_path)
            except Exception as e:
                print(f"[ERROR] {item_path}: {e}")

        print(f"✅ Cleaned: {path}")

In [6231]:
import base64


def to_html(df):
    styled = (
        df.style.hide(axis="index")
        .format(
            lambda x: (
                f"{int(x)}"  # 20.0 -> 20
                if isinstance(x, float) and x.is_integer()
                else (
                    f"{x:.0e}"  # small numbers -> scientific
                    if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01
                    else x
                )
            )
        )
        .set_table_styles(
            [
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {
                    "selector": "th",
                    "props": [
                        ("border-right", "1px solid black"),
                        ("color", "darkblue"),
                        ("font-weight", "bold"),
                        ("padding-left", "8px"),
                        ("padding-right", "8px"),
                    ],
                },
            ]
        )
        .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)


def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = [
        "#0b3d91",  # dark blue
        "#800000",  # maroon
        "#205522",  # dark green
        "#111011",  # indigo
        "#444444",
    ]  # dark gray
    color = colors[(num - 1) % len(colors)]  # cycle through colors

    if gen_report:
        with open("results.html", "a") as f:
            f.write(
                f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n'
            )


def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")


def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')


def write(text):
    html_text = text.replace("\n", "<br>\n")

    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,'
        + encoded
        + '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)

    if os.path.exists(path):
        os.remove(path)

In [6232]:
import os
import shutil


def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [6233]:
metrics = []

In [6234]:
import numpy as np
import pandas as pd
from itertools import product


def _find_elbow_idx(x, y):
    if len(x) <= 2:
        return np.argmax(y)

    x = np.asarray(x)
    y = np.asarray(y)

    x1, y1 = x[0], y[0]
    x2, y2 = x[-1], y[-1]

    numerator = np.abs((x2 - x1) * (y1 - y) - (y2 - y1) * (x1 - x))

    denominator = np.hypot(x2 - x1, y2 - y1)

    distances = numerator / denominator

    return np.argmax(distances)


def extract_maxes_elbow(
    df,
    cols=["message_length", "message_length_tt", "seed"],
    max_col="test_time_self_accuracy",
):
    values = []

    for col in cols:
        values.append(set(df[col]))

    combinations = [list(x) for x in product(*values)]

    maxes = []

    for comb in combinations:

        section = filter_df(
            {col: value for col, value in zip(cols, comb)},
            df,
        )

        if section.empty:
            continue

        # Split this section by learning rate
        elbow_rows = []

        for lr, lr_section in section.groupby("learning_rate_tt"):

            lr_section = lr_section.copy()

            lr_section["num_iterations"] = pd.to_numeric(lr_section["num_iterations"])

            lr_section[max_col] = pd.to_numeric(lr_section[max_col])

            lr_section = lr_section.sort_values("num_iterations")

            x = lr_section["num_iterations"].values
            y = lr_section[max_col].values

            elbow_idx = _find_elbow_idx(x, y)

            elbow_row = lr_section.iloc[elbow_idx]

            elbow_rows.append(elbow_row)

        if len(elbow_rows) == 0:
            continue

        elbow_df = pd.DataFrame(elbow_rows)

        # Among all learning rates, keep the elbow row with the
        # highest accuracy
        best_elbow_row = elbow_df.loc[elbow_df[max_col].idxmax()]

        maxes.append(best_elbow_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)

    return final_df.sort_values(by=cols).reset_index(drop=True)

In [6235]:
def extract_maxes(
    df,
    cols=["message_length", "message_length_tt", "seed"],
    max_col="test_time_mutual_accuracy",
):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[pd.to_numeric(section[max_col]).idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols).reset_index(drop=True)

In [6236]:
def find_best_length_elbow(df):
    x = df["message_length_tt"].to_numpy()

    y = (
        df["test_time_self_accuracy"]
        .str.extract(r"([\d.]+)")
        .astype(float)
        .squeeze()
        .to_numpy()
    )

    idx = _find_elbow_idx(x, y)
    return x[_find_elbow_idx(x, y)], df.loc[idx, "test_time_mutual_accuracy"]

In [6237]:
from itertools import product
import pandas as pd


def mean_and_std(
    df,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
    ],
    metrics=["test_time_self_accuracy", "test_time_mutual_accuracy"] + metrics,
    precision=1,
):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i : i + 3]

        rows.append(
            {col: chunk[col].iloc[0] for col in config_cols}
            |
            # {metric: f"{pd.to_numeric(chunk[metric]).mean() * 100:.1f} ± {pd.to_numeric(chunk[metric]).std() * 100:.1f}" for metric in metrics}
            {
                metric: (
                    None
                    if pd.to_numeric(chunk[metric], errors="coerce").dropna().empty
                    else f"{pd.to_numeric(chunk[metric], errors='coerce').mean():.{precision}f} ± "
                    f"{pd.to_numeric(chunk[metric], errors='coerce').std():.{precision}f}"
                )
                for metric in metrics
            }
        )

    out = pd.DataFrame(rows)
    return out.sort_values(by=config_cols)

In [6238]:
def smooth(values, factor=0.5):
    """Exponential moving average smoothing."""
    smoothed = []
    s = values[0]
    for v in values:
        s = factor * s + (1 - factor) * v
        smoothed.append(s)
    return smoothed

In [6239]:
import matplotlib.pyplot as plt

METHOD_COLORS = {
    "GS-ST": "#8c2d04",
    "REINFORCE": "#1f77b4",
    "VQEL": "#ff7f0e",
    "VQEL + TTA (Batch)": "#2ca02c",
    "VQEL + TTA (Dataset)": "#9467bd",
    "VQEL + TTA (Full)": "#7f7f7f",
    "VQEL + TTA (MG)": "#2ca02c",
    "VQEL + TTS": "#7f7f7f",
    "LR = 1e-4": "#8c2d04",
    "LR = 1e-5": "#7f7f7f",
    "LR = 1e-6": "#2ca02c",
    "Oracle": "#000000",
    "Oracle (MG)": "#000000",
    "Oracle (Full)": "#7f7f7f",
}


def plot(
    dfs,
    labels=(
        "GS-ST",
        "REINFORCE",
        "VQEL",
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle",
        "VQEL + TTS",
    ),
    name="name",
    xcol="message_length_tt",
    xlabel="Test-Time Message Length",
    marker="o",
    smooth_factor=0,
    xticks=None,
    logscale=False,
    base=10,
    grid=True,
):
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

    all_x = []

    for df, label in zip(dfs, labels):
        if label not in METHOD_COLORS:
            raise ValueError(f"No color defined for label: {label}")

        color = METHOD_COLORS[label]

        df = df.copy()

        # Split "mean ± std"
        df[["mean_acc", "std_acc"]] = (
            df["test_time_mutual_accuracy"].str.split("±", expand=True).astype(float)
        )

        df = df.sort_values(xcol)

        x = df[xcol]
        y = df["mean_acc"]
        err = df["std_acc"]

        all_x.extend(x.tolist())

        ax.plot(
            x,
            smooth(y, factor=smooth_factor),
            marker=marker if "Oracle" not in label else None,
            linestyle="-" if "Oracle" not in label else "--",
            linewidth=2,
            color=color,
            label=label,
        )

        ax.fill_between(
            x,
            smooth(y - err, factor=smooth_factor),
            smooth(y + err, factor=smooth_factor),
            color=color,
            alpha=0.2,
            linewidth=0,
        )

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel("OoD Accuracy (%)", fontsize=12)

    if xticks == None:
        xticks = sorted(set(all_x))
    ax.set_xticks(xticks)

    ax.tick_params(axis="both", labelsize=10)

    ax.legend(frameon=False)
    if grid:
        ax.grid(alpha=0.3)

    if logscale:
        ax.set_xscale("log", base=base)
        if base == 2:
            ax.set_xticklabels(xticks)
    fig.tight_layout()

    plt.savefig(
        f"assets/{name}.pdf",
        format="pdf",
        bbox_inches="tight",
    )

    if show_plots:
        plt.show()
    else:
        plt.close(fig)

In [6240]:
clear()

---

In [6241]:
gumbel_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "tau_0",
    "mutual_play_accuracy",
    "sampling_temperature",
    "test_time_mode",
    "path",
]

backbone_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "test_time_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "path",
]

backbone_cols_report = [
    "seed",
    "message_length",
    "agent_a_training_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

baseline_cols = (
    [
        "seed",
        "baseline",
        "dataset",
        "sim",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "test_time_self_accuracy",
        "test_time_mutual_accuracy",
    ]
    + metrics
    + [
        "path",
    ]
)

baseline_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

scaling_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

scaling_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

adapt_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

adapt_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

# Shape

## Gumbel - ID

In [6242]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "None",
        "baseline": True,
        "gumbel": True,
    },
    sort_by=["seed"],
)
# res[gumbel_cols]

In [6243]:
final = extract_maxes(res, max_col="mutual_play_accuracy")
# final[gumbel_cols]

In [6244]:
mean = mean_and_std(
    final, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
to_html(mean)
mean

,dataset,message_length,mutual_play_accuracy
0,shape1,"[3, 4]",84.2 ± 0.8


## Gumbel - OOD

In [6245]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "!None",
        "baseline": True,
        "test_time_mode": "-",
        "gumbel": True,
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6246]:
gumbel = mean_and_std(
    res, metrics=["test_time_self_accuracy", "test_time_mutual_accuracy"]
)
gumbel

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,shape1,"[3, 4]",4,shape2,-,0.0 ± 0.0,38.8 ± 0.7
1,shape1,"[3, 4]",5,shape2,-,0.0 ± 0.0,39.5 ± 0.4
2,shape1,"[3, 4]",6,shape2,-,0.0 ± 0.0,39.0 ± 1.3
3,shape1,"[3, 4]",7,shape2,-,0.0 ± 0.0,39.0 ± 1.2
4,shape1,"[3, 4]",8,shape2,-,0.0 ± 0.0,38.5 ± 1.7
5,shape1,"[3, 4]",9,shape2,-,0.0 ± 0.0,36.8 ± 2.3
6,shape1,"[3, 4]",10,shape2,-,0.0 ± 0.0,34.4 ± 2.5
7,shape1,"[3, 4]",11,shape2,-,0.0 ± 0.0,31.9 ± 2.2
8,shape1,"[3, 4]",12,shape2,-,0.0 ± 0.0,29.8 ± 2.8
9,shape1,"[3, 4]",13,shape2,-,0.0 ± 0.0,27.0 ± 3.0


## REINFORCE - ID

In [6247]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "None",
        "baseline": True,
        "gumbel": False,
    },
    sort_by=["seed"],
)
# res[backbone_cols]

In [6248]:
mean = mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
to_html(mean)
mean

,dataset,message_length,mutual_play_accuracy
0,shape1,"[3, 4]",86.5 ± 0.2


## REINFORCE - OOD

In [6249]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "!None",
        "baseline": True,
        "gumbel": False,
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6250]:
REINFORCE = mean_and_std(res)
REINFORCE

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,shape1,"[3, 4]",4,shape2,-,0.0 ± 0.0,45.5 ± 2.9
1,shape1,"[3, 4]",5,shape2,-,0.0 ± 0.0,46.5 ± 3.0
2,shape1,"[3, 4]",6,shape2,-,0.0 ± 0.0,44.9 ± 3.5
3,shape1,"[3, 4]",7,shape2,-,0.0 ± 0.0,40.6 ± 5.6
4,shape1,"[3, 4]",8,shape2,-,0.0 ± 0.0,35.2 ± 7.1
5,shape1,"[3, 4]",9,shape2,-,0.0 ± 0.0,30.5 ± 9.5
6,shape1,"[3, 4]",10,shape2,-,0.0 ± 0.0,26.6 ± 10.3
7,shape1,"[3, 4]",11,shape2,-,0.0 ± 0.0,23.3 ± 10.5
8,shape1,"[3, 4]",12,shape2,-,0.0 ± 0.0,20.8 ± 10.2
9,shape1,"[3, 4]",13,shape2,-,0.0 ± 0.0,18.7 ± 10.0


## VQEL - ID

In [6251]:
add_heading(2, "Shape1")
add_heading(3, "Base Model")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "baseline": False,
        "message_length": "[3, 4]",
        "test_time_mode": "-",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length"],
)
res = extract_maxes(res, max_col="mutual_play_accuracy")
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,shape1,cosine,"[3, 4]",1e-03,-,1e-04,-,84.1,88.9,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...
1,2,False,shape1,cosine,"[3, 4]",1e-03,-,1e-03,-,83.7,88.8,20251228_1437_bs32_vocab10_repr1024_msg_len10_...
2,3,False,shape1,cosine,"[3, 4]",1e-03,-,1e-04,-,84.6,88.5,20251228_2016_bs32_vocab10_repr1024_msg_len10_...


In [6252]:
mean = mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)
to_html(mean)
mean

,dataset,message_length,self_play_accuracy_a,mutual_play_accuracy
0,shape1,"[3, 4]",84.1 ± 0.5,88.7 ± 0.2


## VQEL - OOD

In [6253]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dataset_tt": "shape2",
        "baseline": False,
        "message_length": "[3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)

to_html(res[baseline_cols_report])

# res[baseline_cols]

In [6254]:
VQ_NoTT = mean_and_std(res)
VQ_NoTT

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,shape1,"[3, 4]",4,shape2,-,41.2 ± 2.5,47.9 ± 1.1
1,shape1,"[3, 4]",5,shape2,-,43.6 ± 2.2,49.6 ± 2.0
2,shape1,"[3, 4]",6,shape2,-,44.1 ± 2.2,50.1 ± 2.2
3,shape1,"[3, 4]",7,shape2,-,44.1 ± 2.9,48.8 ± 3.5
4,shape1,"[3, 4]",8,shape2,-,43.9 ± 2.5,47.6 ± 4.5
5,shape1,"[3, 4]",9,shape2,-,43.3 ± 2.4,46.7 ± 5.1
6,shape1,"[3, 4]",10,shape2,-,42.7 ± 3.4,44.7 ± 5.5
7,shape1,"[3, 4]",11,shape2,-,42.2 ± 2.9,43.8 ± 6.6
8,shape1,"[3, 4]",12,shape2,-,41.1 ± 3.9,42.6 ± 6.9
9,shape1,"[3, 4]",13,shape2,-,40.2 ± 3.4,40.7 ± 7.3


In [6255]:
find_best_length_elbow(VQ_NoTT)

(np.int64(8), '47.6 ± 4.5')

## Scaling

In [6256]:
add_heading(3, "Scaling")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "VQEL": True,
        "message_length": "[3, 4]",
        "test_time_mode": "scaling",
    },
    sort_by=[
        "message_length",
        "message_length_tt",
        "seed",
        "sampling_temperature_tt",
        "best_of_n",
    ],
)

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [6257]:
final = extract_maxes(res)
# final[scaling_cols]

In [6258]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [6259]:
add_heading(3, "Adaptation")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "VQEL": True,
        "message_length": "[3, 4]",
        "test_time_mode": ["batch_adaptation"],
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt"],
)

# res[adapt_cols]

In [6260]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [6261]:
batch_adapt = mean_and_std(final)
batch_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,shape1,"[3, 4]",4,shape2,batch_adaptation,48.9 ± 2.8,52.3 ± 2.1
1,shape1,"[3, 4]",5,shape2,batch_adaptation,52.7 ± 3.9,54.7 ± 2.2
2,shape1,"[3, 4]",6,shape2,batch_adaptation,54.4 ± 3.9,55.4 ± 2.5
3,shape1,"[3, 4]",7,shape2,batch_adaptation,55.6 ± 4.2,55.9 ± 3.1
4,shape1,"[3, 4]",8,shape2,batch_adaptation,55.3 ± 4.9,55.8 ± 2.6
5,shape1,"[3, 4]",9,shape2,batch_adaptation,55.5 ± 4.4,55.0 ± 3.3
6,shape1,"[3, 4]",10,shape2,batch_adaptation,56.5 ± 3.9,55.3 ± 2.2
7,shape1,"[3, 4]",11,shape2,batch_adaptation,54.8 ± 5.3,54.1 ± 3.9
8,shape1,"[3, 4]",12,shape2,batch_adaptation,57.6 ± 3.7,55.4 ± 2.3
9,shape1,"[3, 4]",13,shape2,batch_adaptation,57.0 ± 3.3,55.2 ± 2.8


In [6262]:
find_best_length_elbow(batch_adapt)

(np.int64(7), '55.9 ± 3.1')

## Dataset Adaptation

In [6263]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "VQEL": True,
        "message_length": "[3, 4]",
        "test_time_mode": ["dataset_adaptation"],
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt"],
)
# res[adapt_cols]

In [6264]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [6265]:
dataset_adapt = mean_and_std(final)
# dataset_adapt

In [6266]:
find_best_length_elbow(dataset_adapt)

(np.int64(8), '51.3 ± 2.2')

## Oracle

In [6267]:
res = filter_df(
    {
        "dataset": "shape1",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "shape2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6268]:
final = extract_maxes(res)

In [6269]:
oracle = mean_and_std(final)
oracle

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,shape1,"[3, 4]",4,shape2,oracle_adaptation,0.0 ± 0.0,63.1 ± 2.7
1,shape1,"[3, 4]",5,shape2,oracle_adaptation,0.0 ± 0.0,68.5 ± 1.7
2,shape1,"[3, 4]",6,shape2,oracle_adaptation,0.0 ± 0.0,71.3 ± 1.2
3,shape1,"[3, 4]",7,shape2,oracle_adaptation,0.0 ± 0.0,74.8 ± 2.2
4,shape1,"[3, 4]",8,shape2,oracle_adaptation,0.0 ± 0.0,75.0 ± 2.0
5,shape1,"[3, 4]",9,shape2,oracle_adaptation,0.0 ± 0.0,76.1 ± 2.1
6,shape1,"[3, 4]",10,shape2,oracle_adaptation,0.0 ± 0.0,77.6 ± 3.2
7,shape1,"[3, 4]",11,shape2,oracle_adaptation,0.0 ± 0.0,77.0 ± 2.3
8,shape1,"[3, 4]",12,shape2,oracle_adaptation,0.0 ± 0.0,76.9 ± 2.7
9,shape1,"[3, 4]",13,shape2,oracle_adaptation,0.0 ± 0.0,77.5 ± 2.3


## Oracle Full

In [6270]:
res = filter_df(
    {
        "dataset": "shape1",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "shape2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6271]:
final = extract_maxes(res)

In [6272]:
oracle_full = mean_and_std(final)
oracle_full

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,shape1,"[3, 4]",4,shape2,oracle_full_adaptation,0.0 ± 0.0,83.7 ± 2.7
1,shape1,"[3, 4]",5,shape2,oracle_full_adaptation,0.0 ± 0.0,87.1 ± 0.7
2,shape1,"[3, 4]",6,shape2,oracle_full_adaptation,0.0 ± 0.0,90.0 ± 1.8
3,shape1,"[3, 4]",7,shape2,oracle_full_adaptation,0.0 ± 0.0,91.9 ± 1.2
4,shape1,"[3, 4]",8,shape2,oracle_full_adaptation,0.0 ± 0.0,92.4 ± 1.1
5,shape1,"[3, 4]",9,shape2,oracle_full_adaptation,0.0 ± 0.0,93.4 ± 0.7
6,shape1,"[3, 4]",10,shape2,oracle_full_adaptation,0.0 ± 0.0,94.2 ± 0.7
7,shape1,"[3, 4]",11,shape2,oracle_full_adaptation,0.0 ± 0.0,95.2 ± 0.9
8,shape1,"[3, 4]",12,shape2,oracle_full_adaptation,0.0 ± 0.0,95.0 ± 0.5
9,shape1,"[3, 4]",13,shape2,oracle_full_adaptation,0.0 ± 0.0,95.4 ± 0.5


## Plot

In [6273]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, dataset_adapt, oracle],
    name="main-results/shape",
)

In [6274]:
plot(
    [batch_adapt, dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/shape",
)

In [6275]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, scaling],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/shape",
)

# MNIST

## Gumbel - ID

In [6276]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "None",
        "gumbel": True,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["seed"],
)

# res[gumbel_cols]

In [6277]:
final = extract_maxes(res, max_col="mutual_play_accuracy")
final[gumbel_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase2_a,learning_rate_phase2_b,tau_0,mutual_play_accuracy,sampling_temperature,test_time_mode,path
0,1,True,mnist1,cosine,"[3, 4]",-,1e-05,1e-01,80.8,1e-05,-,20260212_1441_bs32_vocab10_repr192_msg_len4_lr...
1,2,True,mnist1,cosine,"[3, 4]",-,1e-05,1e-01,80.3,1e-05,-,20260216_2326_bs32_vocab10_repr192_msg_len4_lr...
2,3,True,mnist1,cosine,"[3, 4]",-,1e-05,0.3,79.4,1e-05,-,20260217_1444_bs32_vocab10_repr192_msg_len4_lr...


In [6278]:
mean = mean_and_std(
    final, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
to_html(mean)
mean

,dataset,message_length,mutual_play_accuracy
0,mnist1,"[3, 4]",80.2 ± 0.7


## Gumbel - OOD

In [6279]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "!None",
        "gumbel": True,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6280]:
mnist_gumbel = mean_and_std(res)
mnist_gumbel

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",4,mnist2,-,0.0 ± 0.0,23.7 ± 3.3
1,mnist1,"[3, 4]",5,mnist2,-,0.0 ± 0.0,24.5 ± 2.7
2,mnist1,"[3, 4]",6,mnist2,-,0.0 ± 0.0,23.5 ± 4.0
3,mnist1,"[3, 4]",7,mnist2,-,0.0 ± 0.0,21.7 ± 4.0
4,mnist1,"[3, 4]",8,mnist2,-,0.0 ± 0.0,20.2 ± 4.8
5,mnist1,"[3, 4]",9,mnist2,-,0.0 ± 0.0,18.4 ± 4.8
6,mnist1,"[3, 4]",10,mnist2,-,0.0 ± 0.0,16.6 ± 4.7
7,mnist1,"[3, 4]",11,mnist2,-,0.0 ± 0.0,14.3 ± 4.4
8,mnist1,"[3, 4]",12,mnist2,-,0.0 ± 0.0,12.9 ± 4.2
9,mnist1,"[3, 4]",13,mnist2,-,0.0 ± 0.0,11.9 ± 4.4


## REINFORCE - ID

In [6281]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "None",
        "gumbel": False,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["seed"],
)
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,True,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,0,92.5,20251230_1620_bs32_vocab10_repr192_msg_len4_lr...
1,2,True,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,0,85.8,20251230_1705_bs32_vocab10_repr192_msg_len4_lr...
2,3,True,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,0,88.2,20251230_1750_bs32_vocab10_repr192_msg_len4_lr...


In [6282]:
mean_and_std(res)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",4,mnist1,-,0.0 ± 0.0,86.9 ± 2.3


## REINFORCE - OOD

In [6283]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "!None",
        "gumbel": False,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6284]:
mnist_REINFORCE = mean_and_std(res)
mnist_REINFORCE

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",4,mnist2,-,0.0 ± 0.0,25.7 ± 5.0
1,mnist1,"[3, 4]",5,mnist2,-,0.0 ± 0.0,25.7 ± 5.6
2,mnist1,"[3, 4]",6,mnist2,-,0.0 ± 0.0,24.0 ± 6.7
3,mnist1,"[3, 4]",7,mnist2,-,0.0 ± 0.0,21.4 ± 7.4
4,mnist1,"[3, 4]",8,mnist2,-,0.0 ± 0.0,18.6 ± 8.1
5,mnist1,"[3, 4]",9,mnist2,-,0.0 ± 0.0,16.0 ± 7.7
6,mnist1,"[3, 4]",10,mnist2,-,0.0 ± 0.0,14.2 ± 7.2
7,mnist1,"[3, 4]",11,mnist2,-,0.0 ± 0.0,12.8 ± 6.9
8,mnist1,"[3, 4]",12,mnist2,-,0.0 ± 0.0,11.6 ± 6.2
9,mnist1,"[3, 4]",13,mnist2,-,0.0 ± 0.0,10.4 ± 5.4


## VQEL - ID

In [6285]:
add_heading(2, "MNIST")
add_heading(3, "Base Model")
res = filter_df(
    {
        "dataset": "mnist1",
        "sim": "cosine",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": "[3, 4]",
        "number_of_candidates": 100,
    },
    sort_by=["message_length", "message_length_tt"],
)
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,78.8,89.5,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...
1,2,False,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,78.7,89.1,20251228_1548_bs32_vocab10_repr192_msg_len10_l...
2,3,False,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,79.9,91.8,20251228_1637_bs32_vocab10_repr192_msg_len10_l...


In [6286]:
mean = mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)
mean

,dataset,message_length,self_play_accuracy_a,mutual_play_accuracy
0,mnist1,"[3, 4]",79.1 ± 0.7,90.1 ± 1.5


## VQEL - OOD

In [6287]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "mnist1",
        "sim": "cosine",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "-",
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)
# res[baseline_cols]

In [6288]:
final = extract_maxes(res)
# final[baseline_cols]

In [6289]:
mnist_VQ_NoTT = mean_and_std(final)
# mnist_VQ_NoTT

In [6290]:
find_best_length_elbow(mnist_VQ_NoTT)

(np.int64(8), '48.1 ± 2.0')

## Scaling

In [6291]:
add_heading(3, "Scaling")
res = filter_df(
    {
        "dataset": "mnist1",
        "sim": "cosine",
        "baseline": False,
        "test_time_mode": "scaling",
        "sampling_temperature_tt": "1e-02",
        "number_of_candidates": 100,
    },
    sort_by=[
        "message_length",
        "message_length_tt",
        "sampling_temperature_tt",
        "best_of_n",
    ],
)
# res[scaling_cols]

In [6292]:
final = extract_maxes(res)
# final[scaling_cols]

In [6293]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [6294]:
add_heading(3, "Adaptation")

res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)

# res[adapt_cols]

In [6295]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [6296]:
mnist_batch_adapt = mean_and_std(final)
mnist_batch_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",4,mnist2,batch_adaptation,49.0 ± 2.1,45.5 ± 1.5
1,mnist1,"[3, 4]",5,mnist2,batch_adaptation,57.0 ± 3.6,51.1 ± 2.7
2,mnist1,"[3, 4]",6,mnist2,batch_adaptation,60.0 ± 3.1,54.2 ± 2.8
3,mnist1,"[3, 4]",7,mnist2,batch_adaptation,63.8 ± 3.8,57.1 ± 2.7
4,mnist1,"[3, 4]",8,mnist2,batch_adaptation,64.3 ± 1.8,57.6 ± 1.3
5,mnist1,"[3, 4]",9,mnist2,batch_adaptation,63.5 ± 2.2,56.6 ± 2.5
6,mnist1,"[3, 4]",10,mnist2,batch_adaptation,73.3 ± 1.5,59.4 ± 2.6
7,mnist1,"[3, 4]",11,mnist2,batch_adaptation,66.2 ± 3.5,56.1 ± 3.8
8,mnist1,"[3, 4]",12,mnist2,batch_adaptation,65.6 ± 2.6,52.4 ± 2.3
9,mnist1,"[3, 4]",13,mnist2,batch_adaptation,66.5 ± 3.6,51.4 ± 4.1


In [6297]:
find_best_length_elbow(mnist_batch_adapt)

(np.int64(10), '59.4 ± 2.6')

## Dataset Adaptation

In [6298]:
add_heading(3, "Adaptation")

res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)

res[adapt_cols]

,seed,baseline,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,5,49.6,44.6,20260102_2141_bs32_vocab10_repr192_msg_len4_lr...
1,1,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,10,49.3,45.9,20251228_1030_bs32_vocab10_repr192_msg_len4_lr...
2,1,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,15,50.3,45.5,20260102_2222_bs32_vocab10_repr192_msg_len4_lr...
3,1,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,20,50.7,45.2,20260104_1920_bs32_vocab10_repr192_msg_len4_lr...
4,2,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,5,45.4,41.2,20260102_2145_bs32_vocab10_repr192_msg_len4_lr...
5,2,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,10,45.4,41.1,20260102_2113_bs32_vocab10_repr192_msg_len4_lr...
6,2,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,15,45.9,40.8,20260102_2227_bs32_vocab10_repr192_msg_len4_lr...
7,2,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,20,46.4,41.1,20260104_1925_bs32_vocab10_repr192_msg_len4_lr...
8,3,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,5,44.5,40.5,20260102_2148_bs32_vocab10_repr192_msg_len4_lr...
9,3,False,mnist1,cosine,"[3, 4]",4,mnist2,dataset_adaptation,1e-04,10,44.9,40.6,20260102_2117_bs32_vocab10_repr192_msg_len4_lr...


In [6299]:
final = extract_maxes_elbow(res, max_col="test_time_self_accuracy")
# final[adapt_cols]

In [6300]:
mnist_dataset_adapt = mean_and_std(final)
mnist_dataset_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",4,mnist2,dataset_adaptation,46.7 ± 2.2,42.5 ± 2.9
1,mnist1,"[3, 4]",5,mnist2,dataset_adaptation,53.6 ± 2.8,48.2 ± 2.7
2,mnist1,"[3, 4]",6,mnist2,dataset_adaptation,58.5 ± 3.2,52.6 ± 2.7
3,mnist1,"[3, 4]",7,mnist2,dataset_adaptation,61.3 ± 2.6,54.5 ± 3.1
4,mnist1,"[3, 4]",8,mnist2,dataset_adaptation,62.2 ± 1.7,55.7 ± 1.7
5,mnist1,"[3, 4]",9,mnist2,dataset_adaptation,63.7 ± 1.0,56.5 ± 0.8
6,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,67.7 ± 1.1,56.9 ± 0.7
7,mnist1,"[3, 4]",11,mnist2,dataset_adaptation,64.8 ± 1.2,55.5 ± 1.2
8,mnist1,"[3, 4]",12,mnist2,dataset_adaptation,62.8 ± 1.4,54.2 ± 1.2
9,mnist1,"[3, 4]",13,mnist2,dataset_adaptation,65.3 ± 1.1,54.1 ± 1.7


In [6301]:
find_best_length_elbow(mnist_dataset_adapt)

(np.int64(10), '56.9 ± 0.7')

## Oracle

In [6302]:
res = filter_df(
    {
        "dataset": "mnist1",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "mnist2",
        "number_of_candidates": 100,
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6303]:
final = extract_maxes(res)

In [6304]:
oracle = mean_and_std(final)
oracle

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",4,mnist2,oracle_adaptation,0.0 ± 0.0,53.1 ± 1.1
1,mnist1,"[3, 4]",5,mnist2,oracle_adaptation,0.0 ± 0.0,58.7 ± 2.2
2,mnist1,"[3, 4]",6,mnist2,oracle_adaptation,0.0 ± 0.0,63.1 ± 2.4
3,mnist1,"[3, 4]",7,mnist2,oracle_adaptation,0.0 ± 0.0,64.9 ± 2.3
4,mnist1,"[3, 4]",8,mnist2,oracle_adaptation,0.0 ± 0.0,66.0 ± 2.9
5,mnist1,"[3, 4]",9,mnist2,oracle_adaptation,0.0 ± 0.0,67.2 ± 3.4
6,mnist1,"[3, 4]",10,mnist2,oracle_adaptation,0.0 ± 0.0,67.2 ± 2.8
7,mnist1,"[3, 4]",11,mnist2,oracle_adaptation,0.0 ± 0.0,67.6 ± 4.5
8,mnist1,"[3, 4]",12,mnist2,oracle_adaptation,0.0 ± 0.0,66.8 ± 3.6
9,mnist1,"[3, 4]",13,mnist2,oracle_adaptation,0.0 ± 0.0,67.0 ± 3.7


## Oracle Full

In [6305]:
res = filter_df(
    {
        "dataset": "mnist1",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "mnist2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6306]:
final = extract_maxes(res)

In [6307]:
oracle_full = mean_and_std(final)
oracle_full

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",4,mnist2,oracle_full_adaptation,0.0 ± 0.0,77.8 ± 4.5
1,mnist1,"[3, 4]",5,mnist2,oracle_full_adaptation,0.0 ± 0.0,83.7 ± 3.4
2,mnist1,"[3, 4]",6,mnist2,oracle_full_adaptation,0.0 ± 0.0,87.7 ± 3.9
3,mnist1,"[3, 4]",7,mnist2,oracle_full_adaptation,0.0 ± 0.0,88.5 ± 3.1
4,mnist1,"[3, 4]",8,mnist2,oracle_full_adaptation,0.0 ± 0.0,88.1 ± 3.0
5,mnist1,"[3, 4]",9,mnist2,oracle_full_adaptation,0.0 ± 0.0,88.5 ± 1.2
6,mnist1,"[3, 4]",10,mnist2,oracle_full_adaptation,0.0 ± 0.0,89.7 ± 1.0
7,mnist1,"[3, 4]",11,mnist2,oracle_full_adaptation,0.0 ± 0.0,90.3 ± 1.8
8,mnist1,"[3, 4]",12,mnist2,oracle_full_adaptation,0.0 ± 0.0,91.3 ± 2.3
9,mnist1,"[3, 4]",13,mnist2,oracle_full_adaptation,0.0 ± 0.0,91.1 ± 1.3


## Plot

In [6308]:
plot(
    [
        mnist_gumbel,
        mnist_REINFORCE,
        mnist_VQ_NoTT,
        mnist_batch_adapt,
        mnist_dataset_adapt,
        oracle,
    ],
    name="main-results/mnist",
)

In [6309]:
plot(
    [mnist_batch_adapt, mnist_dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/mnist",
)

In [6310]:
plot(
    [mnist_gumbel, mnist_REINFORCE, mnist_VQ_NoTT, mnist_batch_adapt, scaling],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/mnist",
)

# ImageNet

## Gumbel - ID

In [6311]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
        "gumbel": True,
    },
    sort_by=["message_length", "seed"],
)
# res[gumbel_cols]

In [6312]:
final = extract_maxes(res, max_col="mutual_play_accuracy")
final[gumbel_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase2_a,learning_rate_phase2_b,tau_0,mutual_play_accuracy,sampling_temperature,test_time_mode,path
0,1,True,imagenet,cosine,"[2, 3, 4]",-,1e-05,1e-01,72.6,1,-,20260214_1231_bs32_vocab10_repr2048_msg_len4_l...
1,2,True,imagenet,cosine,"[2, 3, 4]",-,1e-05,1e-01,73.6,1e-05,-,20260217_0025_bs32_vocab10_repr2048_msg_len4_l...
2,3,True,imagenet,cosine,"[2, 3, 4]",-,1e-05,1e-01,74.4,1e-05,-,20260217_0215_bs32_vocab10_repr2048_msg_len4_l...


In [6313]:
mean = mean_and_std(
    final, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
mean

,dataset,message_length,mutual_play_accuracy
0,imagenet,"[2, 3, 4]",73.5 ± 0.9


## Gumbel - OOD

In [6314]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "dialogued_checkpoint": "!None",
        "gumbel": True,
        "dataset_tt": "imagenet_same_class",
        "message_length": "[2, 3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6315]:
imagenet_gumbel = mean_and_std(
    res, metrics=["test_time_self_accuracy", "test_time_mutual_accuracy"] + metrics
)
imagenet_gumbel

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,-,0.0 ± 0.0,35.4 ± 1.4
1,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,0.0 ± 0.0,37.6 ± 4.7
2,imagenet,"[2, 3, 4]",6,imagenet_same_class,-,0.0 ± 0.0,37.7 ± 1.1
3,imagenet,"[2, 3, 4]",7,imagenet_same_class,-,0.0 ± 0.0,35.3 ± 1.6
4,imagenet,"[2, 3, 4]",8,imagenet_same_class,-,0.0 ± 0.0,36.2 ± 1.2
5,imagenet,"[2, 3, 4]",9,imagenet_same_class,-,0.0 ± 0.0,34.5 ± 0.7
6,imagenet,"[2, 3, 4]",10,imagenet_same_class,-,0.0 ± 0.0,32.9 ± 0.5
7,imagenet,"[2, 3, 4]",11,imagenet_same_class,-,0.0 ± 0.0,29.3 ± 1.2
8,imagenet,"[2, 3, 4]",12,imagenet_same_class,-,0.0 ± 0.0,27.8 ± 1.8
9,imagenet,"[2, 3, 4]",13,imagenet_same_class,-,0.0 ± 0.0,26.8 ± 3.1


## REINFORCE - ID

In [6316]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
        "gumbel": False,
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,True,imagenet,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,85.9,20251231_1826_bs32_vocab10_repr2048_msg_len4_l...
1,2,True,imagenet,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,84.6,20251231_1934_bs32_vocab10_repr2048_msg_len4_l...
2,3,True,imagenet,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,86.9,20251231_1958_bs32_vocab10_repr2048_msg_len4_l...


In [6317]:
mean_and_std(res)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet,-,0.0 ± 0.0,85.8 ± 1.2


## REINFORCE - OOD

In [6318]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "!None",
        "gumbel": False,
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6319]:
final = extract_maxes(res)
# final[baseline_cols]

In [6320]:
imagenet_REINFORCE = mean_and_std(final)
imagenet_REINFORCE

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,-,0.0 ± 0.0,30.9 ± 1.3
1,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,0.0 ± 0.0,33.7 ± 1.2
2,imagenet,"[2, 3, 4]",6,imagenet_same_class,-,0.0 ± 0.0,36.2 ± 0.6
3,imagenet,"[2, 3, 4]",7,imagenet_same_class,-,0.0 ± 0.0,37.2 ± 2.0
4,imagenet,"[2, 3, 4]",8,imagenet_same_class,-,0.0 ± 0.0,32.3 ± 1.4
5,imagenet,"[2, 3, 4]",9,imagenet_same_class,-,0.0 ± 0.0,31.7 ± 1.1
6,imagenet,"[2, 3, 4]",10,imagenet_same_class,-,0.0 ± 0.0,29.0 ± 2.8
7,imagenet,"[2, 3, 4]",11,imagenet_same_class,-,0.0 ± 0.0,24.4 ± 3.3
8,imagenet,"[2, 3, 4]",12,imagenet_same_class,-,0.0 ± 0.0,25.3 ± 1.9
9,imagenet,"[2, 3, 4]",13,imagenet_same_class,-,0.0 ± 0.0,20.8 ± 1.2


## VQEL - ID

In [6321]:
add_heading(2, "ImageNet")
add_heading(3, "Base Model")

res = filter_df(
    {
        "dataset": "imagenet",
        "sim": "cosine",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": "[2, 3, 4]",
    },
    sort_by=["vocab_size", "message_length", "seed", "message_length_tt"],
)
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,imagenet,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,88.4,89.1,20251231_2027_bs32_vocab10_repr2048_msg_len4_l...
1,2,False,imagenet,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,88.4,90.1,20251231_2046_bs32_vocab10_repr2048_msg_len4_l...
2,3,False,imagenet,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,88.1,89.8,20260101_1131_bs32_vocab10_repr2048_msg_len4_l...


In [6322]:
mean = mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)
# mean

## VQEL - OOD

In [6323]:
add_heading(3, "Baseline")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": "-",
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)
# res[baseline_cols]

In [6324]:
imagenet_VQ_NoTT = mean_and_std(res)
# imagenet_VQ_NoTT

In [6325]:
find_best_length_elbow(imagenet_VQ_NoTT)

(np.int64(6), '43.9 ± 0.8')

## Scaling

In [6326]:
add_heading(3, "Scaling")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": "scaling",
    },
    sort_by=["message_length_tt", "sampling_temperature_tt", "best_of_n"],
)
# res[scaling_cols]

In [6327]:
final = extract_maxes(res)
# final[scaling_cols]

In [6328]:
scaling = mean_and_std(final)
scaling

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,scaling,41.5 ± 1.4,43.4 ± 1.8
1,imagenet,"[2, 3, 4]",5,imagenet_same_class,scaling,44.4 ± 0.7,46.5 ± 0.8
2,imagenet,"[2, 3, 4]",6,imagenet_same_class,scaling,48.5 ± 3.5,45.9 ± 1.6
3,imagenet,"[2, 3, 4]",7,imagenet_same_class,scaling,47.0 ± 4.9,46.2 ± 1.4
4,imagenet,"[2, 3, 4]",8,imagenet_same_class,scaling,46.4 ± 1.8,43.4 ± 2.3
5,imagenet,"[2, 3, 4]",9,imagenet_same_class,scaling,41.2 ± 4.4,42.5 ± 1.7
6,imagenet,"[2, 3, 4]",10,imagenet_same_class,scaling,35.9 ± 8.3,41.3 ± 1.0
7,imagenet,"[2, 3, 4]",11,imagenet_same_class,scaling,39.3 ± 3.7,40.7 ± 2.5
8,imagenet,"[2, 3, 4]",12,imagenet_same_class,scaling,31.8 ± 10.5,36.0 ± 2.0
9,imagenet,"[2, 3, 4]",13,imagenet_same_class,scaling,32.5 ± 2.7,34.3 ± 3.2


## Batch Adaptation

In [6329]:
add_heading(3, "Adaptation")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": ["batch_adaptation"],
        "message_length": "[2, 3, 4]",
        "num_iterations": [50, 100, 200],
    },
    sort_by=[
        "test_time_mode",
        "message_length_tt",
        "seed",
        "learning_rate_tt",
        "num_iterations",
    ],
)
# res[adapt_cols]

In [6330]:
final = extract_maxes_elbow(res, max_col="test_time_mutual_accuracy")
# final[adapt_cols]

In [6331]:
imagenet_batch_adapt = mean_and_std(final)
# imagenet_batch_adapt

In [6332]:
find_best_length_elbow(imagenet_batch_adapt)

(np.int64(6), '71.1 ± 2.1')

## Dataset Adaptation

In [6333]:
add_heading(3, "Adaptation")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": ["dataset_adaptation"],
        "message_length": "[2, 3, 4]",
        "num_iterations": [5, 10, 15, 20],
    },
    sort_by=[
        "test_time_mode",
        "message_length_tt",
        "seed",
        "learning_rate_tt",
        "num_iterations",
    ],
)
# res[adapt_cols]

In [6334]:
final = extract_maxes_elbow(res)
final[adapt_cols]

,seed,baseline,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,imagenet,cosine,"[2, 3, 4]",4,imagenet_same_class,dataset_adaptation,1e-04,15,66.6,63,20260102_2236_bs32_vocab10_repr2048_msg_len4_l...
1,2,False,imagenet,cosine,"[2, 3, 4]",4,imagenet_same_class,dataset_adaptation,1e-04,10,67.5,62.7,20260102_2128_bs32_vocab10_repr2048_msg_len4_l...
2,3,False,imagenet,cosine,"[2, 3, 4]",4,imagenet_same_class,dataset_adaptation,1e-04,10,70.4,68.9,20260102_2135_bs32_vocab10_repr2048_msg_len4_l...
3,1,False,imagenet,cosine,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,1e-04,10,74.4,69.6,20260525_1903_bs32_vocab10_repr2048_msg_len5_m...
4,2,False,imagenet,cosine,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,1e-04,10,77.6,69.1,20260525_1904_bs32_vocab10_repr2048_msg_len5_m...
5,3,False,imagenet,cosine,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,1e-04,10,76.1,69.6,20260525_1904_bs32_vocab10_repr2048_msg_len5_m...
6,1,False,imagenet,cosine,"[2, 3, 4]",6,imagenet_same_class,dataset_adaptation,1e-04,10,72.9,66.9,20260102_2122_bs32_vocab10_repr2048_msg_len6_l...
7,2,False,imagenet,cosine,"[2, 3, 4]",6,imagenet_same_class,dataset_adaptation,1e-04,15,75.7,68.1,20260102_2245_bs32_vocab10_repr2048_msg_len6_l...
8,3,False,imagenet,cosine,"[2, 3, 4]",6,imagenet_same_class,dataset_adaptation,1e-04,10,74.9,67.6,20260102_2136_bs32_vocab10_repr2048_msg_len6_l...
9,1,False,imagenet,cosine,"[2, 3, 4]",7,imagenet_same_class,dataset_adaptation,1e-04,15,72.5,62,20260102_2238_bs32_vocab10_repr2048_msg_len7_l...


In [6335]:
imagenet_dataset_adapt = mean_and_std(final)
imagenet_dataset_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,dataset_adaptation,68.2 ± 2.0,64.9 ± 3.5
1,imagenet,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,76.0 ± 1.6,69.4 ± 0.3
2,imagenet,"[2, 3, 4]",6,imagenet_same_class,dataset_adaptation,74.5 ± 1.4,67.5 ± 0.6
3,imagenet,"[2, 3, 4]",7,imagenet_same_class,dataset_adaptation,74.5 ± 1.7,65.2 ± 3.0
4,imagenet,"[2, 3, 4]",8,imagenet_same_class,dataset_adaptation,74.8 ± 0.8,61.6 ± 3.6
5,imagenet,"[2, 3, 4]",9,imagenet_same_class,dataset_adaptation,71.9 ± 1.5,55.4 ± 4.0
6,imagenet,"[2, 3, 4]",10,imagenet_same_class,dataset_adaptation,71.7 ± 1.7,54.3 ± 2.9
7,imagenet,"[2, 3, 4]",11,imagenet_same_class,dataset_adaptation,71.6 ± 1.5,53.1 ± 1.9
8,imagenet,"[2, 3, 4]",12,imagenet_same_class,dataset_adaptation,71.9 ± 3.3,51.8 ± 3.6
9,imagenet,"[2, 3, 4]",13,imagenet_same_class,dataset_adaptation,72.4 ± 0.9,48.8 ± 1.0


In [6336]:
find_best_length_elbow(imagenet_dataset_adapt)

(np.int64(5), '69.4 ± 0.3')

## Oracle

In [6337]:
res = filter_df(
    {
        "dataset": "imagenet",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6338]:
final = extract_maxes(res)

In [6339]:
oracle = mean_and_std(final)
oracle

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,86.0 ± 0.7
1,imagenet,"[2, 3, 4]",5,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,90.0 ± 0.4
2,imagenet,"[2, 3, 4]",6,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,91.0 ± 0.6
3,imagenet,"[2, 3, 4]",7,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,91.6 ± 0.6
4,imagenet,"[2, 3, 4]",8,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,91.6 ± 0.3
5,imagenet,"[2, 3, 4]",9,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,91.6 ± 1.0
6,imagenet,"[2, 3, 4]",10,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,91.7 ± 0.6
7,imagenet,"[2, 3, 4]",11,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,91.0 ± 0.6
8,imagenet,"[2, 3, 4]",12,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,90.9 ± 0.4
9,imagenet,"[2, 3, 4]",13,imagenet_same_class,oracle_adaptation,0.0 ± 0.0,90.3 ± 1.0


## Oracle Full

In [6340]:
res = filter_df(
    {
        "dataset": "imagenet",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6341]:
final = extract_maxes(res)

In [6342]:
oracle_full = mean_and_std(final)
oracle_full

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,94.6 ± 0.2
1,imagenet,"[2, 3, 4]",5,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,96.9 ± 0.7
2,imagenet,"[2, 3, 4]",6,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,96.8 ± 0.7
3,imagenet,"[2, 3, 4]",7,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,97.5 ± 0.5
4,imagenet,"[2, 3, 4]",8,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,97.7 ± 1.0
5,imagenet,"[2, 3, 4]",9,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,98.1 ± 0.8
6,imagenet,"[2, 3, 4]",10,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,98.3 ± 0.3
7,imagenet,"[2, 3, 4]",11,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,98.3 ± 0.4
8,imagenet,"[2, 3, 4]",12,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,98.2 ± 0.4
9,imagenet,"[2, 3, 4]",13,imagenet_same_class,oracle_full_adaptation,0.0 ± 0.0,98.8 ± 0.3


## Plot

In [6343]:
plot(
    [
        imagenet_gumbel,
        imagenet_REINFORCE,
        imagenet_VQ_NoTT,
        imagenet_batch_adapt,
        imagenet_dataset_adapt,
        oracle,
    ],
    name="main-results/imagenet",
)

In [6344]:
plot(
    [imagenet_batch_adapt, imagenet_dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/imagenet",
)

In [6345]:
plot(
    [
        imagenet_gumbel,
        imagenet_REINFORCE,
        imagenet_VQ_NoTT,
        imagenet_batch_adapt,
        scaling,
    ],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/imagenet",
)

# COCO

## Gumbel - ID

In [6346]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": True,
        "gumbel": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length", "seed"],
)
# res[gumbel_cols]

In [6347]:
final = extract_maxes(res)
final[gumbel_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase2_a,learning_rate_phase2_b,tau_0,mutual_play_accuracy,sampling_temperature,test_time_mode,path
0,1,True,coco,cosine,"[2, 3, 4]",-,1e-05,1e-01,86.1,1e-05,-,20260609_1520_bs32_vocab10_repr768_msg_len4_ms...
1,2,True,coco,cosine,"[2, 3, 4]",-,1e-05,1e-01,85.5,1e-05,-,20260609_1539_bs32_vocab10_repr768_msg_len4_ms...
2,3,True,coco,cosine,"[2, 3, 4]",-,1e-04,0.2,86.5,1e-05,-,20260610_1218_bs32_vocab10_repr768_msg_len4_ms...


In [6348]:
mean_and_std(final)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco,-,0.0 ± 0.0,86.0 ± 0.5


## Gumbel - OOD

In [6349]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": True,
        "gumbel": True,
        "dialogued_checkpoint": "!None",
        "dataset_tt": "coco_complex",
        "message_length": "[2, 3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6350]:
gumbel = mean_and_std(res)
gumbel

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco_complex,-,0.0 ± 0.0,40.9 ± 0.4
1,coco,"[2, 3, 4]",5,coco_complex,-,0.0 ± 0.0,41.7 ± 0.7
2,coco,"[2, 3, 4]",6,coco_complex,-,0.0 ± 0.0,41.8 ± 1.4
3,coco,"[2, 3, 4]",7,coco_complex,-,0.0 ± 0.0,40.8 ± 2.8
4,coco,"[2, 3, 4]",8,coco_complex,-,0.0 ± 0.0,38.5 ± 3.6
5,coco,"[2, 3, 4]",9,coco_complex,-,0.0 ± 0.0,36.7 ± 4.7
6,coco,"[2, 3, 4]",10,coco_complex,-,0.0 ± 0.0,34.1 ± 5.5
7,coco,"[2, 3, 4]",11,coco_complex,-,0.0 ± 0.0,31.7 ± 6.9
8,coco,"[2, 3, 4]",12,coco_complex,-,0.0 ± 0.0,29.7 ± 7.9
9,coco,"[2, 3, 4]",13,coco_complex,-,0.0 ± 0.0,27.9 ± 8.5


## REINFORCE - ID

In [6351]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": True,
        "gumbel": False,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length", "seed"],
)
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,True,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,88.9,20260609_1605_bs32_vocab10_repr768_msg_len4_ms...
1,1,True,coco,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,85,20260609_1614_bs32_vocab10_repr768_msg_len4_ms...
2,2,True,coco,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,85.3,20260609_1631_bs32_vocab10_repr768_msg_len4_ms...
3,2,True,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,88.4,20260609_1623_bs32_vocab10_repr768_msg_len4_ms...
4,3,True,coco,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,83.2,20260609_1649_bs32_vocab10_repr768_msg_len4_ms...
5,3,True,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,89.1,20260609_1640_bs32_vocab10_repr768_msg_len4_ms...


In [6352]:
final = extract_maxes(res)
final[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,True,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,88.9,20260609_1605_bs32_vocab10_repr768_msg_len4_ms...
1,2,True,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,88.4,20260609_1623_bs32_vocab10_repr768_msg_len4_ms...
2,3,True,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,89.1,20260609_1640_bs32_vocab10_repr768_msg_len4_ms...


In [6353]:
mean_and_std(final)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco,-,0.0 ± 0.0,88.8 ± 0.4


## REINFORCE - OOD

In [6354]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": True,
        "gumbel": False,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "!None",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [6355]:
REINFORCE = mean_and_std(res)
REINFORCE

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco_complex,-,0.0 ± 0.0,42.9 ± 1.3
1,coco,"[2, 3, 4]",5,coco_complex,-,0.0 ± 0.0,45.7 ± 1.4
2,coco,"[2, 3, 4]",6,coco_complex,-,0.0 ± 0.0,44.4 ± 2.0
3,coco,"[2, 3, 4]",7,coco_complex,-,0.0 ± 0.0,40.2 ± 1.0
4,coco,"[2, 3, 4]",8,coco_complex,-,0.0 ± 0.0,33.8 ± 1.2
5,coco,"[2, 3, 4]",9,coco_complex,-,0.0 ± 0.0,29.0 ± 1.1
6,coco,"[2, 3, 4]",10,coco_complex,-,0.0 ± 0.0,23.8 ± 1.6
7,coco,"[2, 3, 4]",11,coco_complex,-,0.0 ± 0.0,19.9 ± 1.8
8,coco,"[2, 3, 4]",12,coco_complex,-,0.0 ± 0.0,17.6 ± 1.2
9,coco,"[2, 3, 4]",13,coco_complex,-,0.0 ± 0.0,15.7 ± 1.2


## VQEL - ID

In [6356]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco",
        "baseline": False,
        "message_length": "[2, 3, 4]",
    },
    sort_by=["seed"],
)
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,coco,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,87.7,89.6,20260609_1409_bs32_vocab10_repr768_msg_len4_ms...
1,1,False,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,86.7,89.9,20260609_1419_bs32_vocab10_repr768_msg_len4_ms...
2,2,False,coco,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,87.5,88.8,20260609_1429_bs32_vocab10_repr768_msg_len4_ms...
3,2,False,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,85.9,89.6,20260609_1440_bs32_vocab10_repr768_msg_len4_ms...
4,3,False,coco,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,87.9,88.3,20260609_1450_bs32_vocab10_repr768_msg_len4_ms...
5,3,False,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,83.8,88.2,20260609_1501_bs32_vocab10_repr768_msg_len4_ms...


In [6357]:
final = extract_maxes(res)
final[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,86.7,89.9,20260609_1419_bs32_vocab10_repr768_msg_len4_ms...
1,2,False,coco,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,85.9,89.6,20260609_1440_bs32_vocab10_repr768_msg_len4_ms...
2,3,False,coco,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,87.9,88.3,20260609_1450_bs32_vocab10_repr768_msg_len4_ms...


In [6358]:
mean_and_std(final)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco,-,86.8 ± 1.0,89.3 ± 0.9


## VQEL - OOD

In [6359]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": False,
        "test_time_mode": "-",
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)
# res[baseline_cols]

In [6360]:
VQ_NoTT = mean_and_std(res)
VQ_NoTT

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco_complex,-,69.0 ± 2.1,49.7 ± 1.3
1,coco,"[2, 3, 4]",5,coco_complex,-,68.9 ± 2.0,50.7 ± 1.1
2,coco,"[2, 3, 4]",6,coco_complex,-,69.9 ± 1.5,51.3 ± 0.9
3,coco,"[2, 3, 4]",7,coco_complex,-,68.4 ± 2.0,50.3 ± 1.2
4,coco,"[2, 3, 4]",8,coco_complex,-,67.4 ± 1.8,50.4 ± 1.6
5,coco,"[2, 3, 4]",9,coco_complex,-,65.5 ± 1.9,49.8 ± 1.5
6,coco,"[2, 3, 4]",10,coco_complex,-,63.4 ± 2.5,48.9 ± 1.3
7,coco,"[2, 3, 4]",11,coco_complex,-,60.7 ± 2.3,48.3 ± 1.2
8,coco,"[2, 3, 4]",12,coco_complex,-,57.8 ± 3.2,47.7 ± 2.8
9,coco,"[2, 3, 4]",13,coco_complex,-,55.4 ± 3.0,47.1 ± 2.9


In [6361]:
find_best_length_elbow(VQ_NoTT)

(np.int64(8), '50.4 ± 1.6')

## Scaling

In [6362]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": False,
        "message_length": "[2, 3, 4]",
        "test_time_mode": "scaling",
    },
    sort_by=["message_length_tt", "seed", "sampling_temperature_tt"],
)
# res[scaling_cols]

In [6363]:
final = extract_maxes(res)
# final[scaling_cols]

In [6364]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [6365]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": False,
        "test_time_mode": ["batch_adaptation"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)
# res[adapt_cols]

In [6366]:
final = extract_maxes_elbow(res)
final[adapt_cols]

,seed,baseline,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,coco,cosine,"[2, 3, 4]",4,coco_complex,batch_adaptation,1e-04,100,67.7,63.6,20260610_0056_bs32_vocab10_repr768_msg_len4_ms...
1,2,False,coco,cosine,"[2, 3, 4]",4,coco_complex,batch_adaptation,1e-04,50,67.4,64.4,20260610_0002_bs32_vocab10_repr768_msg_len4_ms...
2,3,False,coco,cosine,"[2, 3, 4]",4,coco_complex,batch_adaptation,1e-04,100,67.3,62.9,20260610_0059_bs32_vocab10_repr768_msg_len4_ms...
3,1,False,coco,cosine,"[2, 3, 4]",5,coco_complex,batch_adaptation,1e-04,100,71.3,65.4,20260610_0101_bs32_vocab10_repr768_msg_len5_ms...
4,2,False,coco,cosine,"[2, 3, 4]",5,coco_complex,batch_adaptation,1e-04,100,72.7,66.2,20260610_0103_bs32_vocab10_repr768_msg_len5_ms...
5,3,False,coco,cosine,"[2, 3, 4]",5,coco_complex,batch_adaptation,1e-04,100,72.2,65.6,20260610_0104_bs32_vocab10_repr768_msg_len5_ms...
6,1,False,coco,cosine,"[2, 3, 4]",6,coco_complex,batch_adaptation,1e-04,50,71.7,67.4,20260610_0006_bs32_vocab10_repr768_msg_len6_ms...
7,2,False,coco,cosine,"[2, 3, 4]",6,coco_complex,batch_adaptation,1e-04,100,74.1,66.3,20260610_0108_bs32_vocab10_repr768_msg_len6_ms...
8,3,False,coco,cosine,"[2, 3, 4]",6,coco_complex,batch_adaptation,1e-04,100,74.1,65.7,20260610_0110_bs32_vocab10_repr768_msg_len6_ms...
9,1,False,coco,cosine,"[2, 3, 4]",7,coco_complex,batch_adaptation,1e-04,50,71.1,67,20260610_0010_bs32_vocab10_repr768_msg_len7_ms...


In [6367]:
batch_adapt = mean_and_std(final)
batch_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco_complex,batch_adaptation,67.5 ± 0.2,63.6 ± 0.8
1,coco,"[2, 3, 4]",5,coco_complex,batch_adaptation,72.1 ± 0.7,65.7 ± 0.4
2,coco,"[2, 3, 4]",6,coco_complex,batch_adaptation,73.3 ± 1.4,66.5 ± 0.9
3,coco,"[2, 3, 4]",7,coco_complex,batch_adaptation,72.9 ± 1.7,66.2 ± 0.9
4,coco,"[2, 3, 4]",8,coco_complex,batch_adaptation,73.6 ± 1.4,64.5 ± 1.9
5,coco,"[2, 3, 4]",9,coco_complex,batch_adaptation,72.7 ± 1.6,64.0 ± 1.3
6,coco,"[2, 3, 4]",10,coco_complex,batch_adaptation,73.8 ± 2.8,62.8 ± 2.2
7,coco,"[2, 3, 4]",11,coco_complex,batch_adaptation,74.0 ± 2.5,62.1 ± 2.8
8,coco,"[2, 3, 4]",12,coco_complex,batch_adaptation,73.3 ± 3.1,60.8 ± 3.0
9,coco,"[2, 3, 4]",13,coco_complex,batch_adaptation,74.8 ± 0.9,59.5 ± 2.9


In [6368]:
find_best_length_elbow(batch_adapt)

(np.int64(6), '66.5 ± 0.9')

## Dataset Adaptation

In [6369]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": False,
        "test_time_mode": ["dataset_adaptation"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)
# res[adapt_cols]

In [6370]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [6371]:
dataset_adapt = mean_and_std(final)
dataset_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco_complex,dataset_adaptation,57.7 ± 0.8,57.7 ± 0.3
1,coco,"[2, 3, 4]",5,coco_complex,dataset_adaptation,61.8 ± 1.7,59.6 ± 1.0
2,coco,"[2, 3, 4]",6,coco_complex,dataset_adaptation,63.1 ± 2.1,59.8 ± 1.0
3,coco,"[2, 3, 4]",7,coco_complex,dataset_adaptation,63.4 ± 3.2,58.3 ± 1.7
4,coco,"[2, 3, 4]",8,coco_complex,dataset_adaptation,64.0 ± 1.9,57.9 ± 2.2
5,coco,"[2, 3, 4]",9,coco_complex,dataset_adaptation,63.0 ± 3.4,56.6 ± 2.1
6,coco,"[2, 3, 4]",10,coco_complex,dataset_adaptation,61.4 ± 1.3,55.6 ± 2.5
7,coco,"[2, 3, 4]",11,coco_complex,dataset_adaptation,61.2 ± 0.6,55.0 ± 3.3
8,coco,"[2, 3, 4]",12,coco_complex,dataset_adaptation,62.2 ± 3.7,54.2 ± 3.3
9,coco,"[2, 3, 4]",13,coco_complex,dataset_adaptation,61.4 ± 2.8,53.9 ± 3.4


In [6372]:
find_best_length_elbow(dataset_adapt)

(np.int64(8), '57.9 ± 2.2')

## Oracle

In [6373]:
res = filter_df(
    {
        "dataset": "coco",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "coco_complex",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6374]:
final = extract_maxes(res)

In [6375]:
oracle = mean_and_std(final)
oracle

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco_complex,oracle_adaptation,0.0 ± 0.0,68.9 ± 1.9
1,coco,"[2, 3, 4]",5,coco_complex,oracle_adaptation,0.0 ± 0.0,72.0 ± 1.9
2,coco,"[2, 3, 4]",6,coco_complex,oracle_adaptation,0.0 ± 0.0,73.3 ± 1.7
3,coco,"[2, 3, 4]",7,coco_complex,oracle_adaptation,0.0 ± 0.0,73.7 ± 2.1
4,coco,"[2, 3, 4]",8,coco_complex,oracle_adaptation,0.0 ± 0.0,73.6 ± 1.2
5,coco,"[2, 3, 4]",9,coco_complex,oracle_adaptation,0.0 ± 0.0,73.3 ± 1.3
6,coco,"[2, 3, 4]",10,coco_complex,oracle_adaptation,0.0 ± 0.0,73.1 ± 2.0
7,coco,"[2, 3, 4]",11,coco_complex,oracle_adaptation,0.0 ± 0.0,73.1 ± 1.2
8,coco,"[2, 3, 4]",12,coco_complex,oracle_adaptation,0.0 ± 0.0,71.3 ± 2.5
9,coco,"[2, 3, 4]",13,coco_complex,oracle_adaptation,0.0 ± 0.0,70.3 ± 3.0


## Oracle Full

In [6376]:
res = filter_df(
    {
        "dataset": "coco",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "coco_complex",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6377]:
final = extract_maxes(res)

In [6378]:
oracle_full = mean_and_std(final)
oracle_full

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,coco,"[2, 3, 4]",4,coco_complex,oracle_full_adaptation,0.0 ± 0.0,92.3 ± 1.2
1,coco,"[2, 3, 4]",5,coco_complex,oracle_full_adaptation,0.0 ± 0.0,94.3 ± 0.7
2,coco,"[2, 3, 4]",6,coco_complex,oracle_full_adaptation,0.0 ± 0.0,95.4 ± 0.2
3,coco,"[2, 3, 4]",7,coco_complex,oracle_full_adaptation,0.0 ± 0.0,95.7 ± 0.9
4,coco,"[2, 3, 4]",8,coco_complex,oracle_full_adaptation,0.0 ± 0.0,96.1 ± 0.2
5,coco,"[2, 3, 4]",9,coco_complex,oracle_full_adaptation,0.0 ± 0.0,96.9 ± 0.5
6,coco,"[2, 3, 4]",10,coco_complex,oracle_full_adaptation,0.0 ± 0.0,97.2 ± 0.2
7,coco,"[2, 3, 4]",11,coco_complex,oracle_full_adaptation,0.0 ± 0.0,96.8 ± 0.5
8,coco,"[2, 3, 4]",12,coco_complex,oracle_full_adaptation,0.0 ± 0.0,96.7 ± 0.5
9,coco,"[2, 3, 4]",13,coco_complex,oracle_full_adaptation,0.0 ± 0.0,97.1 ± 0.7


## Plot

In [6379]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, dataset_adapt, oracle],
    name="main-results/coco",
)

In [6380]:
plot(
    [batch_adapt, dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/coco",
)

In [6381]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, scaling],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/coco",
)

# Visual Genome

## Gumbel - ID

In [6382]:
res = filter_df(
    {
        "dataset": "genome",
        "baseline": True,
        "gumbel": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length", "seed"],
)
# res[gumbel_cols]

In [6383]:
final = extract_maxes(res)
final[gumbel_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase2_a,learning_rate_phase2_b,tau_0,mutual_play_accuracy,sampling_temperature,test_time_mode,path
0,1,True,genome,cosine,"[2, 3, 4]",-,1e-05,1e-01,61.5,1e-05,-,20260925_0059_bs32_vocab10_repr2048_msg_len4_m...
1,2,True,genome,cosine,"[2, 3, 4]",-,1e-05,1e-01,59.9,1e-05,-,20260925_0321_bs32_vocab10_repr2048_msg_len4_m...
2,3,True,genome,cosine,"[2, 3, 4]",-,1e-05,1e-01,62.9,1e-05,-,20260925_0556_bs32_vocab10_repr2048_msg_len4_m...


In [6384]:
for i in range(3):
    print(final.loc[i, "path"])

20260925_0059_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1
20260925_0321_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed2
20260925_0556_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed3


In [6385]:
mean_and_std(final)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,genome,"[2, 3, 4]",4,genome,-,0.0 ± 0.0,61.4 ± 1.5


## Gumbel - OOD

In [6386]:
res = filter_df(
    {
        "dataset": "genome",
        "baseline": True,
        "gumbel": True,
        "dialogued_checkpoint": "!None",
        "dataset_tt": "genome_complex",
        "message_length": "[2, 3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)
res[baseline_cols]

,seed,baseline,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,True,genome,cosine,"[2, 3, 4]",4,genome_complex,-,0,50.5,20260925_0929_bs32_vocab10_repr2048_msg_len4_m...
1,2,True,genome,cosine,"[2, 3, 4]",4,genome_complex,-,0,50.9,20260925_0929_bs32_vocab10_repr2048_msg_len4_m...
2,3,True,genome,cosine,"[2, 3, 4]",4,genome_complex,-,0,54.3,20260925_0930_bs32_vocab10_repr2048_msg_len4_m...
3,1,True,genome,cosine,"[2, 3, 4]",5,genome_complex,-,0,50.8,20260925_1016_bs32_vocab10_repr2048_msg_len5_m...
4,2,True,genome,cosine,"[2, 3, 4]",5,genome_complex,-,0,51.8,20260925_1017_bs32_vocab10_repr2048_msg_len5_m...
5,3,True,genome,cosine,"[2, 3, 4]",5,genome_complex,-,0,55.3,20260925_1017_bs32_vocab10_repr2048_msg_len5_m...
6,1,True,genome,cosine,"[2, 3, 4]",6,genome_complex,-,0,50.1,20260925_1017_bs32_vocab10_repr2048_msg_len6_m...
7,2,True,genome,cosine,"[2, 3, 4]",6,genome_complex,-,0,52.1,20260925_1017_bs32_vocab10_repr2048_msg_len6_m...
8,3,True,genome,cosine,"[2, 3, 4]",6,genome_complex,-,0,55.9,20260925_1017_bs32_vocab10_repr2048_msg_len6_m...
9,1,True,genome,cosine,"[2, 3, 4]",7,genome_complex,-,0,49.3,20260925_1017_bs32_vocab10_repr2048_msg_len7_m...


In [6387]:
gumbel = mean_and_std(res)
gumbel

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,genome,"[2, 3, 4]",4,genome_complex,-,0.0 ± 0.0,51.9 ± 2.1
1,genome,"[2, 3, 4]",5,genome_complex,-,0.0 ± 0.0,52.6 ± 2.4
2,genome,"[2, 3, 4]",6,genome_complex,-,0.0 ± 0.0,52.7 ± 2.9
3,genome,"[2, 3, 4]",7,genome_complex,-,0.0 ± 0.0,52.4 ± 3.1
4,genome,"[2, 3, 4]",8,genome_complex,-,0.0 ± 0.0,49.9 ± 2.8
5,genome,"[2, 3, 4]",9,genome_complex,-,0.0 ± 0.0,48.1 ± 3.2
6,genome,"[2, 3, 4]",10,genome_complex,-,0.0 ± 0.0,44.6 ± 2.5
7,genome,"[2, 3, 4]",11,genome_complex,-,0.0 ± 0.0,41.3 ± 2.2
8,genome,"[2, 3, 4]",12,genome_complex,-,0.0 ± 0.0,38.3 ± 2.0
9,genome,"[2, 3, 4]",13,genome_complex,-,0.0 ± 0.0,35.9 ± 2.5


## REINFORCE - ID

In [6388]:
res = filter_df(
    {
        "dataset": "genome",
        "baseline": True,
        "gumbel": False,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length", "seed"],
)
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,True,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,5.9,20260925_0000_bs32_vocab10_repr2048_msg_len4_m...
1,1,True,genome,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,0,9.2,20260925_0039_bs32_vocab10_repr2048_msg_len4_m...
2,1,True,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,5.7,20260924_2352_bs32_vocab10_repr2048_msg_len4_m...
3,1,True,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,6.5,20260925_0020_bs32_vocab10_repr2048_msg_len4_m...
4,2,True,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,7.5,20260925_0223_bs32_vocab10_repr2048_msg_len4_m...
5,2,True,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,4.7,20260925_0242_bs32_vocab10_repr2048_msg_len4_m...
6,2,True,genome,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,0,6.3,20260925_0302_bs32_vocab10_repr2048_msg_len4_m...
7,3,True,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,0,6.5,20260925_0517_bs32_vocab10_repr2048_msg_len4_m...
8,3,True,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,8.4,20260925_0458_bs32_vocab10_repr2048_msg_len4_m...
9,3,True,genome,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,0,5.9,20260925_0536_bs32_vocab10_repr2048_msg_len4_m...


In [6389]:
final = extract_maxes(res)
final[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,True,genome,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,0,9.2,20260925_0039_bs32_vocab10_repr2048_msg_len4_m...
1,2,True,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,7.5,20260925_0223_bs32_vocab10_repr2048_msg_len4_m...
2,3,True,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,0,8.4,20260925_0458_bs32_vocab10_repr2048_msg_len4_m...


In [6390]:
mean_and_std(final)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,genome,"[2, 3, 4]",4,genome,-,0.0 ± 0.0,8.4 ± 0.9


## REINFORCE - OOD

## VQEL - ID

In [6391]:
res = filter_df(
    {
        "dataset": "genome",
        "dataset_tt": "genome",
        "baseline": False,
        "message_length": "[2, 3, 4]",
    },
    sort_by=["seed"],
)
res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,genome,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,54.6,57.9,20260925_0211_bs32_vocab10_repr2048_msg_len4_m...
1,1,False,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,55.3,61.8,20260925_0159_bs32_vocab10_repr2048_msg_len4_m...
2,1,False,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,36.7,50.5,20260924_2322_bs32_vocab10_repr2048_msg_len4_m...
3,2,False,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,51.1,60.5,20260925_0434_bs32_vocab10_repr2048_msg_len4_m...
4,2,False,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,40.7,52.7,20260925_0421_bs32_vocab10_repr2048_msg_len4_m...
5,2,False,genome,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,58.5,60.5,20260925_0446_bs32_vocab10_repr2048_msg_len4_m...
6,3,False,genome,cosine,"[2, 3, 4]",1e-05,-,1e-05,-,38.3,50.9,20260925_0656_bs32_vocab10_repr2048_msg_len4_m...
7,3,False,genome,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,55.4,58.3,20260925_0721_bs32_vocab10_repr2048_msg_len4_m...
8,3,False,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,53.2,61.8,20260925_0709_bs32_vocab10_repr2048_msg_len4_m...


In [6392]:
final = extract_maxes(res)
final[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,55.3,61.8,20260925_0159_bs32_vocab10_repr2048_msg_len4_m...
1,2,False,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,51.1,60.5,20260925_0434_bs32_vocab10_repr2048_msg_len4_m...
2,3,False,genome,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,53.2,61.8,20260925_0709_bs32_vocab10_repr2048_msg_len4_m...


In [6393]:
for i in range(3):
    print(final.loc[i, "path"])

20260925_0159_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_0.0001_lr2a_0.0001_lr2b_0.0001_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1
20260925_0434_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_0.0001_lr2a_0.0001_lr2b_0.0001_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed2
20260925_0709_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_0.0001_lr2a_0.0001_lr2b_0.0001_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed3


In [6394]:
mean_and_std(final)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,genome,"[2, 3, 4]",4,genome,-,54.0 ± 0.2,61.4 ± 0.8


## VQEL - OOD

In [6395]:
res = filter_df(
    {
        "dataset": "genome",
        "dataset_tt": "genome_complex",
        "baseline": False,
        "test_time_mode": "-",
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)
res[baseline_cols]

,seed,baseline,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,genome,cosine,"[2, 3, 4]",4,genome_complex,-,47.1,54.1,20260925_0933_bs32_vocab10_repr2048_msg_len4_m...
1,2,False,genome,cosine,"[2, 3, 4]",4,genome_complex,-,45.3,50.4,20260925_0934_bs32_vocab10_repr2048_msg_len4_m...
2,3,False,genome,cosine,"[2, 3, 4]",4,genome_complex,-,46.8,55.3,20260925_0934_bs32_vocab10_repr2048_msg_len4_m...
3,1,False,genome,cosine,"[2, 3, 4]",5,genome_complex,-,49.1,57.5,20260925_0935_bs32_vocab10_repr2048_msg_len5_m...
4,2,False,genome,cosine,"[2, 3, 4]",5,genome_complex,-,47,55.7,20260925_0936_bs32_vocab10_repr2048_msg_len5_m...
5,3,False,genome,cosine,"[2, 3, 4]",5,genome_complex,-,49.3,58.1,20260925_0936_bs32_vocab10_repr2048_msg_len5_m...
6,1,False,genome,cosine,"[2, 3, 4]",6,genome_complex,-,46.6,57.3,20260925_0936_bs32_vocab10_repr2048_msg_len6_m...
7,2,False,genome,cosine,"[2, 3, 4]",6,genome_complex,-,42.6,56.3,20260925_0936_bs32_vocab10_repr2048_msg_len6_m...
8,3,False,genome,cosine,"[2, 3, 4]",6,genome_complex,-,47.4,58.1,20260925_0936_bs32_vocab10_repr2048_msg_len6_m...
9,1,False,genome,cosine,"[2, 3, 4]",7,genome_complex,-,43.7,54.7,20260925_0936_bs32_vocab10_repr2048_msg_len7_m...


In [6396]:
VQ_NoTT = mean_and_std(res)
VQ_NoTT

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,genome,"[2, 3, 4]",4,genome_complex,-,46.4 ± 1.0,53.3 ± 2.6
1,genome,"[2, 3, 4]",5,genome_complex,-,48.5 ± 1.3,57.1 ± 1.2
2,genome,"[2, 3, 4]",6,genome_complex,-,45.5 ± 2.6,57.2 ± 0.9
3,genome,"[2, 3, 4]",7,genome_complex,-,41.1 ± 3.3,54.7 ± 1.2
4,genome,"[2, 3, 4]",8,genome_complex,-,34.2 ± 6.1,52.5 ± 1.6
5,genome,"[2, 3, 4]",9,genome_complex,-,25.5 ± 8.5,47.6 ± 1.9
6,genome,"[2, 3, 4]",10,genome_complex,-,17.9 ± 10.2,42.9 ± 0.9
7,genome,"[2, 3, 4]",11,genome_complex,-,15.1 ± 8.7,38.0 ± 0.9
8,genome,"[2, 3, 4]",12,genome_complex,-,13.4 ± 6.5,33.8 ± 2.3
9,genome,"[2, 3, 4]",13,genome_complex,-,12.2 ± 4.6,29.8 ± 2.7


## Batch Adaptation

In [6397]:
res = filter_df(
    {
        "dataset": "genome",
        "dataset_tt": "genome_complex",
        "baseline": False,
        "test_time_mode": ["batch_adaptation"],
        "message_length": "[2, 3, 4]",
        "learning_rate_tt": "1e-05"
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)
# res[adapt_cols]

In [6398]:
final = extract_maxes_elbow(res)
final[adapt_cols]

,seed,baseline,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,genome,cosine,"[2, 3, 4]",4,genome_complex,batch_adaptation,1e-05,25,52,57.7,20260925_1134_bs32_vocab10_repr2048_msg_len4_m...
1,2,False,genome,cosine,"[2, 3, 4]",4,genome_complex,batch_adaptation,1e-05,25,50.7,54.7,20260925_1134_bs32_vocab10_repr2048_msg_len4_m...
2,3,False,genome,cosine,"[2, 3, 4]",4,genome_complex,batch_adaptation,1e-05,25,53.5,58.9,20260925_1135_bs32_vocab10_repr2048_msg_len4_m...
3,1,False,genome,cosine,"[2, 3, 4]",5,genome_complex,batch_adaptation,1e-05,25,56.5,61.2,20260925_1135_bs32_vocab10_repr2048_msg_len5_m...
4,2,False,genome,cosine,"[2, 3, 4]",5,genome_complex,batch_adaptation,1e-05,25,54.6,59.4,20260925_1135_bs32_vocab10_repr2048_msg_len5_m...
5,3,False,genome,cosine,"[2, 3, 4]",5,genome_complex,batch_adaptation,1e-05,25,56.9,61.8,20260925_1136_bs32_vocab10_repr2048_msg_len5_m...
6,1,False,genome,cosine,"[2, 3, 4]",6,genome_complex,batch_adaptation,1e-05,25,56.9,63.2,20260925_1136_bs32_vocab10_repr2048_msg_len6_m...
7,2,False,genome,cosine,"[2, 3, 4]",6,genome_complex,batch_adaptation,1e-05,25,53.5,60.6,20260925_1136_bs32_vocab10_repr2048_msg_len6_m...
8,3,False,genome,cosine,"[2, 3, 4]",6,genome_complex,batch_adaptation,1e-05,25,54.7,63.3,20260925_1137_bs32_vocab10_repr2048_msg_len6_m...
9,1,False,genome,cosine,"[2, 3, 4]",7,genome_complex,batch_adaptation,1e-05,25,54.9,62,20260925_1137_bs32_vocab10_repr2048_msg_len7_m...


In [6399]:
batch_adapt = mean_and_std(final)
batch_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,genome,"[2, 3, 4]",4,genome_complex,batch_adaptation,52.1 ± 1.4,57.1 ± 2.2
1,genome,"[2, 3, 4]",5,genome_complex,batch_adaptation,56.0 ± 1.2,60.8 ± 1.2
2,genome,"[2, 3, 4]",6,genome_complex,batch_adaptation,55.0 ± 1.7,62.4 ± 1.5
3,genome,"[2, 3, 4]",7,genome_complex,batch_adaptation,52.3 ± 2.4,61.0 ± 1.5
4,genome,"[2, 3, 4]",8,genome_complex,batch_adaptation,46.3 ± 5.7,58.5 ± 2.2
5,genome,"[2, 3, 4]",9,genome_complex,batch_adaptation,37.3 ± 9.4,55.9 ± 2.6
6,genome,"[2, 3, 4]",10,genome_complex,batch_adaptation,29.4 ± 12.0,51.8 ± 3.0
7,genome,"[2, 3, 4]",11,genome_complex,batch_adaptation,25.5 ± 11.7,47.8 ± 3.1
8,genome,"[2, 3, 4]",12,genome_complex,batch_adaptation,24.5 ± 9.3,43.6 ± 3.0
9,genome,"[2, 3, 4]",13,genome_complex,batch_adaptation,23.9 ± 7.6,41.0 ± 2.8


## Dataset Adaptation

In [6400]:
res = filter_df(
    {
        "dataset": "genome",
        "dataset_tt": "genome_complex",
        "baseline": False,
        "test_time_mode": ["dataset_adaptation"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)
# res[adapt_cols]

In [6401]:
final = extract_maxes_elbow(res)
final[adapt_cols]

,seed,baseline,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,genome,cosine,"[2, 3, 4]",4,genome_complex,dataset_adaptation,1e-05,10,51.8,55.9,20260925_1015_bs32_vocab10_repr2048_msg_len4_m...
1,2,False,genome,cosine,"[2, 3, 4]",4,genome_complex,dataset_adaptation,1e-05,10,51.5,53.3,20260925_1015_bs32_vocab10_repr2048_msg_len4_m...
2,3,False,genome,cosine,"[2, 3, 4]",4,genome_complex,dataset_adaptation,1e-05,15,52.9,56.7,20260925_1016_bs32_vocab10_repr2048_msg_len4_m...
3,1,False,genome,cosine,"[2, 3, 4]",5,genome_complex,dataset_adaptation,1e-05,10,56.7,59.7,20260925_0953_bs32_vocab10_repr2048_msg_len5_m...
4,2,False,genome,cosine,"[2, 3, 4]",5,genome_complex,dataset_adaptation,1e-05,10,55.9,59.3,20260925_0953_bs32_vocab10_repr2048_msg_len5_m...
5,3,False,genome,cosine,"[2, 3, 4]",5,genome_complex,dataset_adaptation,1e-05,10,58.3,59.8,20260925_0953_bs32_vocab10_repr2048_msg_len5_m...
6,1,False,genome,cosine,"[2, 3, 4]",6,genome_complex,dataset_adaptation,1e-05,5,56.3,61.7,20260925_0945_bs32_vocab10_repr2048_msg_len6_m...
7,2,False,genome,cosine,"[2, 3, 4]",6,genome_complex,dataset_adaptation,1e-05,10,58.1,60.3,20260925_0954_bs32_vocab10_repr2048_msg_len6_m...
8,3,False,genome,cosine,"[2, 3, 4]",6,genome_complex,dataset_adaptation,1e-05,10,55.9,62.1,20260925_0954_bs32_vocab10_repr2048_msg_len6_m...
9,1,False,genome,cosine,"[2, 3, 4]",7,genome_complex,dataset_adaptation,1e-05,10,58.3,60.5,20260925_0954_bs32_vocab10_repr2048_msg_len7_m...


In [6402]:
dataset_adapt = mean_and_std(final)
dataset_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,genome,"[2, 3, 4]",4,genome_complex,dataset_adaptation,52.1 ± 0.7,55.3 ± 1.8
1,genome,"[2, 3, 4]",5,genome_complex,dataset_adaptation,57.0 ± 1.2,59.6 ± 0.3
2,genome,"[2, 3, 4]",6,genome_complex,dataset_adaptation,56.8 ± 1.2,61.4 ± 0.9
3,genome,"[2, 3, 4]",7,genome_complex,dataset_adaptation,55.8 ± 2.2,61.5 ± 1.2
4,genome,"[2, 3, 4]",8,genome_complex,dataset_adaptation,53.3 ± 3.5,59.3 ± 2.5
5,genome,"[2, 3, 4]",9,genome_complex,dataset_adaptation,51.5 ± 5.8,56.3 ± 2.5
6,genome,"[2, 3, 4]",10,genome_complex,dataset_adaptation,49.4 ± 7.5,50.9 ± 1.8
7,genome,"[2, 3, 4]",11,genome_complex,dataset_adaptation,47.9 ± 8.7,49.1 ± 1.2
8,genome,"[2, 3, 4]",12,genome_complex,dataset_adaptation,48.5 ± 6.7,46.7 ± 2.3
9,genome,"[2, 3, 4]",13,genome_complex,dataset_adaptation,47.8 ± 5.8,45.0 ± 2.2


## Plot

In [6403]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, dataset_adapt, oracle],
    name="main-results/genome",
)

# Batch Size

In [6404]:
adapt_cols = [
    "seed",
    "baseline",
    "dataset",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "number_of_candidates",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

baseline_cols = [
    "seed",
    "baseline",
    "dataset",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "number_of_candidates",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

## ImageNet

In [6405]:
number_of_candidates = [2, 4, 8, 16, 32, 64, 118]

### Gumbel - OOD

In [6406]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "message_length_tt": 6,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=["message_length_tt", "number_of_candidates", "seed"],
)

# res[baseline_cols]

In [6407]:
gumbel = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### REINFORCE - OOD

In [6408]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "message_length_tt": 7,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=["message_length_tt", "number_of_candidates", "seed"],
)

# res[baseline_cols]

In [6409]:
REINFORCE = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### VQEL - OOD

In [6410]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": False,
        "test_time_mode": "-",
        "message_length_tt": 6,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)

# res[adapt_cols]

In [6411]:
vqel = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)
# vqel

### Batch Adaptation

In [6412]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "number_of_candidates": number_of_candidates,
        "message_length_tt": 5,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [6413]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [6414]:
batch_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Dataset Adaptation

In [6415]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "number_of_candidates": number_of_candidates,
        "message_length_tt": 5,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [6416]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [6417]:
dataset_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Oracle

In [6418]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "test_time_mode": "oracle_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
res[adapt_cols]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,2,1e-04,20,0,100,20260620_1332_bs32_vocab10_repr2048_msg_len15_...
1,2,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,2,1e-04,20,0,100,20260620_1332_bs32_vocab10_repr2048_msg_len15_...
2,3,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,2,1e-04,20,0,100,20260620_1333_bs32_vocab10_repr2048_msg_len15_...
3,1,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,4,1e-04,20,0,100,20260620_1334_bs32_vocab10_repr2048_msg_len15_...
4,2,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,4,1e-04,20,0,100,20260620_1335_bs32_vocab10_repr2048_msg_len15_...
5,3,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,4,1e-04,20,0,100,20260620_1336_bs32_vocab10_repr2048_msg_len15_...
6,1,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,8,1e-04,20,0,99.6,20260620_1336_bs32_vocab10_repr2048_msg_len15_...
7,2,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,8,1e-04,20,0,99.6,20260620_1337_bs32_vocab10_repr2048_msg_len15_...
8,3,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,8,1e-04,20,0,100,20260620_1338_bs32_vocab10_repr2048_msg_len15_...
9,1,True,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,16,1e-04,20,0,96.5,20260620_1339_bs32_vocab10_repr2048_msg_len15_...


In [6419]:
final = extract_maxes(res, cols=["number_of_candidates", "seed"])

In [6420]:
oracle = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)
oracle

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,2,0.0 ± 0.0,100.0 ± 0.0
1,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,4,0.0 ± 0.0,100.0 ± 0.0
2,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,8,0.0 ± 0.0,99.7 ± 0.2
3,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,16,0.0 ± 0.0,97.4 ± 0.9
4,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,32,0.0 ± 0.0,90.4 ± 1.8
5,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,64,0.0 ± 0.0,82.5 ± 0.9
6,imagenet,"[2, 3, 4]",15,imagenet_dog_breed,oracle_adaptation,118,0.0 ± 0.0,72.7 ± 1.6


### Plot

In [6421]:
plot(
    [gumbel, REINFORCE, vqel, batch_adapt, dataset_adapt, oracle],
    labels=[
        "GS-ST",
        "REINFORCE",
        "VQEL",
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle",
    ],
    xlabel="Number of Test-Time Distractors",
    xcol="number_of_candidates",
    logscale=True,
    base=2,
    name="batch/imagenet",
    xticks=[0, 0, 1, 3, 7, 15, 31, 63, 127],
)

/tmp/ipykernel_3321/4247277450.py:102: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(xticks)


## MNIST

In [6422]:
number_of_candidates = [2, 4, 8, 16, 32, 64, 128]

### Gumbel - OOD

In [6423]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "message_length_tt": 4,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [6424]:
gumbel = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### REINFORCE - OOD

In [6425]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "message_length_tt": 5,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [6426]:
REINFORCE = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### VQEL - OOD

In [6427]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "-",
        "message_length_tt": 7,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)

# res[adapt_cols]

In [6428]:
final = extract_maxes(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [6429]:
vqel = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Batch Adaptation

In [6430]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)

# res[adapt_cols]

In [6431]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [6432]:
batch_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Dataset Adaptation

In [6433]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [6434]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
final[adapt_cols]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,2,1e-04,15,100,99.5,20260315_1543_bs32_vocab10_repr192_msg_len10_m...
1,2,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,2,1e-04,15,100,99.7,20260315_1555_bs32_vocab10_repr192_msg_len10_m...
2,3,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,2,1e-04,25,100,99.5,20260315_1914_bs32_vocab10_repr192_msg_len10_m...
3,1,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,4,1e-04,20,99.5,97,20260315_1746_bs32_vocab10_repr192_msg_len10_m...
4,2,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,4,1e-04,25,99.7,97.7,20260315_1943_bs32_vocab10_repr192_msg_len10_m...
5,3,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,4,1e-04,15,98.4,96.1,20260315_1632_bs32_vocab10_repr192_msg_len10_m...
6,1,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,8,1e-04,15,95.6,91,20260315_1638_bs32_vocab10_repr192_msg_len10_m...
7,2,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,8,1e-04,20,97.1,93.2,20260315_1814_bs32_vocab10_repr192_msg_len10_m...
8,3,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,8,1e-04,15,95.2,91.1,20260315_1644_bs32_vocab10_repr192_msg_len10_m...
9,1,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,16,1e-04,20,90.3,83.6,20260315_1822_bs32_vocab10_repr192_msg_len10_m...


In [6435]:
dataset_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Oracle

In [6436]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "test_time_mode": "oracle_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [6437]:
final = extract_maxes(res, cols=["number_of_candidates", "seed"])
final[adapt_cols]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,2,1e-04,20,0,100,20260620_1152_bs32_vocab10_repr192_msg_len15_m...
1,2,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,2,1e-04,20,0,100,20260620_1210_bs32_vocab10_repr192_msg_len15_m...
2,3,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,2,1e-04,20,0,99.9,20260620_1228_bs32_vocab10_repr192_msg_len15_m...
3,1,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,4,1e-03,30,0,98.9,20260621_1040_bs32_vocab10_repr192_msg_len15_m...
4,2,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,4,1e-03,30,0,98.7,20260621_1051_bs32_vocab10_repr192_msg_len15_m...
5,3,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,4,1e-04,20,0,98.2,20260620_1301_bs32_vocab10_repr192_msg_len15_m...
6,1,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,8,1e-03,30,0,97.9,20260621_1113_bs32_vocab10_repr192_msg_len15_m...
7,2,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,8,1e-03,30,0,96.9,20260621_1118_bs32_vocab10_repr192_msg_len15_m...
8,3,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,8,1e-03,20,0,94.6,20260620_1134_bs32_vocab10_repr192_msg_len15_m...
9,1,True,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,16,1e-03,30,0,95.9,20260621_1129_bs32_vocab10_repr192_msg_len15_m...


In [6438]:
oracle = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)
oracle

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy
0,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,2,0.0 ± 0.0,100.0 ± 0.1
1,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,4,0.0 ± 0.0,98.6 ± 0.4
2,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,8,0.0 ± 0.0,96.5 ± 1.7
3,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,16,0.0 ± 0.0,93.2 ± 2.6
4,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,32,0.0 ± 0.0,85.9 ± 3.1
5,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,64,0.0 ± 0.0,74.9 ± 3.9
6,mnist1,"[3, 4]",15,mnist2,oracle_adaptation,128,0.0 ± 0.0,62.4 ± 4.7


### Plot

In [6439]:
plot(
    [gumbel, REINFORCE, vqel, batch_adapt, dataset_adapt, oracle],
    labels=[
        "REINFORCE",
        "GS-ST",
        "VQEL",
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle",
    ],
    xlabel="Number of Test-Time Distractors",
    xcol="number_of_candidates",
    logscale=True,
    base=2,
    name="batch/mnist",
    xticks=[0, 0, 1, 3, 7, 15, 31, 63, 127],
)

/tmp/ipykernel_3321/4247277450.py:102: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(xticks)


# Learning Rate & Steps

In [6440]:
num_iterations = (
    list(range(1, 10)) + list(range(10, 100, 10)) + list(range(100, 1000, 100)) + [1000]
)

## MNIST

In [6441]:
no_adapt = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": ["-"],
        "message_length_tt": 10,
        "learning_rate_tt": "-",
        "number_of_candidates": 100,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
# no_adapt[adapt_cols]

### Batch Adaptation

#### LR = 1e-4

In [6442]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": ["1e-04"],
        "number_of_candidates": 100,
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6443]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6444]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### LR = 1e-5

In [6445]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": ["1e-05"],
        "number_of_candidates": 100,
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6446]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6447]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr5

#### LR = 1e-6

In [6448]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 10,
        "number_of_candidates": 100,
        "learning_rate_tt": ["1e-06"],
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6449]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [6450]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### Plot

In [6451]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=True, xticks=range(0, 1010, 100), smooth_factor=0.5, name="steps/mnist_steps_batch")

### Dataset Adaptation

#### LR = 1e-4

In [6452]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": "1e-04",
        "number_of_candidates": 100,
        "path": ">20260300",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6453]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6454]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr4

#### LR = 1e-5

In [6455]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": "1e-05",
        "number_of_candidates": 100,
        "path": ">20260300",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6456]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6457]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### LR = 1e-6

In [6458]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": "1e-06",
        "number_of_candidates": 100,
        "path": ">20260300",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6459]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6460]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### Plot

In [6461]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=False, smooth_factor=0.5, xticks=range(0, 201, 25), name="steps/mnist_steps_dataset")

## ImageNet

In [6462]:
no_adapt = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": ["-"],
        "message_length_tt": 5,
        "learning_rate_tt": ["-"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
# no_adapt[adapt_cols]

### Batch Adaptation

#### LR = 1e-4

In [6463]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": ["1e-04"],
        "message_length": "[2, 3, 4]",
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6464]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6465]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### LR = 1e-5

In [6466]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": "1e-05",
        "message_length": "[2, 3, 4]",
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6467]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [6468]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr5

#### LR = 1e-6

In [6469]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "dataset_tt": "imagenet_same_class",
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": "1e-06",
        "message_length": "[2, 3, 4]",
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6470]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [6471]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr6

#### Plot

In [6472]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=True, xticks=range(0, 1010, 100), smooth_factor=0.5, name="steps/imagenet_steps_batch")

### Dataset Adaptation

#### LR = 1e-4

In [6473]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "dataset_tt": "imagenet_same_class",
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": ["1e-04"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6474]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6475]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr4

#### LR = 1e-5

In [6476]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "dataset_tt": "imagenet_same_class",
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": ["1e-05"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [6477]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [6478]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr5

#### LR = 1e-6

In [6479]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 5,
        "dataset_tt": "imagenet_same_class",
        "learning_rate_tt": ["1e-06"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])

# res[adapt_cols]

In [6480]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [6481]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr6

#### Plot

In [6482]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=False, smooth_factor=0.6, xticks=range(0, 201, 25), name="steps/imagenet_steps_dataset")

# Computation Costs

In [6483]:
mask = df["inference_time"] != "None"
df.loc[mask, "inference_time"] = df.loc[mask, "inference_time"] / (1000)  # S

mask = df["peak_memory"] != "None"
df.loc[mask, "peak_memory"] = df.loc[mask, "peak_memory"] / (1024**3)  # GB

mask = df["flops"] != "None"
df.loc[mask, "flops"] = df.loc[mask, "flops"] / (10**9)  # GFLOPS

In [6484]:
computation_metrics = [
    "inference_time",
    "peak_memory",
    "flops",
]

## MNIST

### Gumbel

In [6485]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": True,
        "message_length_tt": 10,
        "flops": "!None",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)

res[baseline_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,1,True,mnist1,"[3, 4]",10,mnist2,-,100,0,22.1,20260222_1711_bs32_vocab10_repr192_msg_len10_l...,0.2736895752,0.08895587921,4.1236608
1,2,True,mnist1,"[3, 4]",10,mnist2,-,100,0,13.6,20260222_1713_bs32_vocab10_repr192_msg_len10_l...,0.2499892731,0.08895587921,4.1236608
2,3,True,mnist1,"[3, 4]",10,mnist2,-,100,0,14.2,20260222_1715_bs32_vocab10_repr192_msg_len10_l...,0.2404720001,0.08895587921,4.1236608


In [6486]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,mnist1,"[3, 4]",10,mnist2,-,16.6 ± 4.7,0.3 ± 0.0,0.1 ± 0.0,4.1 ± 0.0


### REINFORCE

In [6487]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": False,
        "message_length_tt": 10,
        "flops": "!None",
    }
)

res[baseline_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,2,True,mnist1,"[3, 4]",10,mnist2,-,100,0,7.3,20251230_2039_bs32_vocab10_repr192_msg_len10_l...,0.2429607086,0.08895301819,4.1236608
1,1,True,mnist1,"[3, 4]",10,mnist2,-,100,0,21.6,20251230_2037_bs32_vocab10_repr192_msg_len10_l...,0.2439423981,0.08895301819,4.1236608
2,3,True,mnist1,"[3, 4]",10,mnist2,-,100,0,13.6,20251230_2040_bs32_vocab10_repr192_msg_len10_l...,0.2439713593,0.08895301819,4.1236608


In [6488]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,mnist1,"[3, 4]",10,mnist2,-,14.2 ± 7.2,0.2 ± 0.0,0.1 ± 0.0,4.1 ± 0.0


### VQEL

In [6489]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 10,
        "test_time_mode": "-",
        "flops": "!None",
    }
)

res[baseline_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,3,False,mnist1,"[3, 4]",10,mnist2,-,100,44.4,42.4,20260316_1906_bs32_vocab10_repr192_msg_len10_m...,0.2891046143,0.08967065811,4.12608
1,2,False,mnist1,"[3, 4]",10,mnist2,-,100,50.9,43,20251229_0028_bs32_vocab10_repr192_msg_len10_l...,0.2924754333,0.08967065811,4.12608
2,1,False,mnist1,"[3, 4]",10,mnist2,-,100,47.5,43.5,20260316_1906_bs32_vocab10_repr192_msg_len10_m...,0.299907135,0.08967065811,4.12608


In [6490]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,mnist1,"[3, 4]",10,mnist2,-,43.0 ± 0.6,0.3 ± 0.0,0.1 ± 0.0,4.1 ± 0.0


### Batch Adaptation

In [6491]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 10,
        "test_time_mode": "batch_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,2,False,mnist1,"[3, 4]",10,mnist2,batch_adaptation,100,1e-04,100,66.5,57.6,20260525_1940_bs32_vocab10_repr192_msg_len10_m...,83.01031472,0.6367726326,17.59296
1,3,False,mnist1,"[3, 4]",10,mnist2,batch_adaptation,100,1e-04,100,63.2,56.2,20260525_1942_bs32_vocab10_repr192_msg_len10_m...,83.6767073,0.6367726326,17.59296
2,1,False,mnist1,"[3, 4]",10,mnist2,batch_adaptation,100,1e-04,100,67.7,58.4,20260525_1939_bs32_vocab10_repr192_msg_len10_m...,96.25747958,0.6367726326,17.59296


In [6492]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,mnist1,"[3, 4]",10,mnist2,batch_adaptation,57.4 ± 1.1,87.6 ± 7.5,0.6 ± 0.0,17.6 ± 0.0


### Dataset Adaptation

In [6493]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 10,
        "test_time_mode": "dataset_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,2,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,100,1e-04,10,56.8,56.1,20260525_1822_bs32_vocab10_repr192_msg_len10_m...,10.09257617,0.3433170319,5.472768
1,1,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,100,1e-04,10,55.2,55.7,20260525_1822_bs32_vocab10_repr192_msg_len10_m...,9.698804687,0.3433170319,5.472768
2,3,False,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,100,1e-04,10,51.7,54.7,20260525_1823_bs32_vocab10_repr192_msg_len10_m...,9.530878906,0.3433170319,5.472768


In [6494]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,mnist1,"[3, 4]",10,mnist2,dataset_adaptation,55.5 ± 0.7,9.8 ± 0.3,0.3 ± 0.0,5.5 ± 0.0


## ImageNet

### Gumbel

In [6495]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": True,
        "message_length_tt": 5,
        "flops": "!None",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)

res[baseline_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,1,True,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,0,40.4,20260222_1717_bs32_vocab10_repr2048_msg_len5_l...,0.3862895508,0.5644178391,0.809238528
1,2,True,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,0,40.1,20260222_1720_bs32_vocab10_repr2048_msg_len5_l...,0.2555688019,0.5644178391,0.809238528
2,3,True,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,0,32.2,20260222_1724_bs32_vocab10_repr2048_msg_len5_l...,0.3410332031,0.5644178391,0.809238528


In [6496]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,37.6 ± 4.7,0.3 ± 0.1,0.6 ± 0.0,0.8 ± 0.0


### REINFORCE

In [6497]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": False,
        "message_length_tt": 5,
        "flops": "!None",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)

res[baseline_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,1,True,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,0,32.4,20260101_1821_bs32_vocab10_repr2048_msg_len5_l...,0.2550014038,0.5644016266,0.809238528
1,2,True,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,0,34.7,20260101_1824_bs32_vocab10_repr2048_msg_len5_l...,0.2555042877,0.5644016266,0.809238528
2,3,True,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,0,33.9,20260101_1826_bs32_vocab10_repr2048_msg_len5_l...,0.4746801453,0.5644016266,0.809238528


In [6498]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,33.7 ± 1.2,0.3 ± 0.1,0.6 ± 0.0,0.8 ± 0.0


### VQEL

In [6499]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "test_time_mode": "-",
        "message_length_tt": 5,
        "flops": "!None",
    }
)

res[baseline_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,2,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,41.2,43.2,20260101_1816_bs32_vocab10_repr2048_msg_len5_l...,0.316145813,0.5656247139,0.812909804
1,1,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,41,44.4,20260101_1813_bs32_vocab10_repr2048_msg_len5_l...,0.3081534424,0.5656247139,0.812909804
2,3,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,32,40.3,43.3,20260101_1818_bs32_vocab10_repr2048_msg_len5_l...,0.3116493835,0.5656247139,0.812909804


In [6500]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,imagenet,"[2, 3, 4]",5,imagenet_same_class,-,43.6 ± 0.7,0.3 ± 0.0,0.6 ± 0.0,0.8 ± 0.0


### Batch Adaptation

In [6501]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 5,
        "test_time_mode": "batch_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,2,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,batch_adaptation,32,1e-04,10,71.1,69.1,20260525_1918_bs32_vocab10_repr2048_msg_len5_m...,12.15374523,1.81649828,25.18054312
1,3,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,batch_adaptation,32,1e-04,10,71.3,68.1,20260525_1919_bs32_vocab10_repr2048_msg_len5_m...,12.04735528,1.81649828,25.18054312
2,1,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,batch_adaptation,32,1e-04,10,72.4,69.6,20260525_1918_bs32_vocab10_repr2048_msg_len5_m...,12.14755176,1.81649828,25.18054312


In [6502]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,imagenet,"[2, 3, 4]",5,imagenet_same_class,batch_adaptation,68.9 ± 0.8,12.1 ± 0.1,1.8 ± 0.0,25.2 ± 0.0


### Dataset Adaptation

In [6503]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 5,
        "test_time_mode": "dataset_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path,inference_time,peak_memory,flops
0,1,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,32,1e-04,10,74.4,69.6,20260525_1903_bs32_vocab10_repr2048_msg_len5_m...,12.11384668,1.201653957,25.18054312
1,2,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,32,1e-04,10,77.6,69.1,20260525_1904_bs32_vocab10_repr2048_msg_len5_m...,12.02887598,1.201653957,25.18054312
2,3,False,imagenet,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,32,1e-04,10,76.1,69.6,20260525_1904_bs32_vocab10_repr2048_msg_len5_m...,12.31262109,1.201653957,25.18054312


In [6504]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,inference_time,peak_memory,flops
0,imagenet,"[2, 3, 4]",5,imagenet_same_class,dataset_adaptation,69.4 ± 0.3,12.2 ± 0.1,1.2 ± 0.0,25.2 ± 0.0


# MPs Alignment

In [6505]:
alignment_metrics = [
    "mp_similarity",
    "mp_similarity_baseline_mean",
    "mp_similarity_baseline_std",
    "mp_similarity_p_value",
]

## Shape

In [6506]:
res = filter_df(
    {
        "dataset": "shape1",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,1,False,shape1,cosine,"[3, 4]",1e-03,-,1e-04,-,84.1,88.9,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...,0.9482,2e-03,7e-04,1e-03
1,3,False,shape1,cosine,"[3, 4]",1e-03,-,1e-04,-,84.6,88.5,20251228_2016_bs32_vocab10_repr1024_msg_len10_...,0.930111,3e-03,7e-04,1e-03
2,2,False,shape1,cosine,"[3, 4]",1e-03,-,1e-03,-,83.7,88.8,20251228_1437_bs32_vocab10_repr1024_msg_len10_...,0.959573,3e-03,7e-04,1e-03


In [6507]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,shape1,"[3, 4]",5,one_shape,-,61.567 ± 23.880,0.946 ± 0.015,0.003 ± 0.001,0.001 ± 0.000,0.001 ± 0.000


## MNIST

In [6508]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,1,False,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,78.8,89.5,20251224_1857_bs32_vocab10_repr192_lr1_0.0001_...,0.966477,4e-03,7e-04,1e-03
1,3,False,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,79.9,91.8,20251229_0029_bs32_vocab10_repr192_msg_len4_lr...,0.968395,4e-03,7e-04,1e-03
2,2,False,mnist1,cosine,"[3, 4]",1e-04,-,1e-04,-,78.7,89.1,20251229_0027_bs32_vocab10_repr192_msg_len4_lr...,0.96118,4e-03,7e-04,1e-03


In [6509]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,mnist1,"[3, 4]",4,mnist2,-,42.033 ± 2.811,0.965 ± 0.004,0.004 ± 0.000,0.001 ± 0.000,0.001 ± 0.000


## ImageNet

In [6510]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,1,False,imagenet,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,88.4,89.1,20260101_1813_bs32_vocab10_repr2048_msg_len4_l...,0.966477,4e-03,7e-04,1e-03
1,3,False,imagenet,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,88.1,89.8,20260101_1818_bs32_vocab10_repr2048_msg_len4_l...,0.942136,8e-03,8e-04,1e-03
2,2,False,imagenet,cosine,"[2, 3, 4]",1e-04,-,1e-04,-,88.4,90.1,20260101_1815_bs32_vocab10_repr2048_msg_len4_l...,0.941884,7e-03,7e-04,1e-03


In [6511]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,-,42.733 ± 2.079,0.950 ± 0.014,0.006 ± 0.002,0.001 ± 0.000,0.001 ± 0.000


## COCO

In [6512]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,1,False,coco,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,87.7,89.6,20260609_1409_bs32_vocab10_repr768_msg_len4_ms...,0.929759,5e-03,7e-04,1e-03
1,3,False,coco,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,87.9,88.3,20260609_1450_bs32_vocab10_repr768_msg_len4_ms...,0.922851,5e-03,7e-04,1e-03
2,2,False,coco,cosine,"[2, 3, 4]",1e-03,-,1e-03,-,87.5,88.8,20260609_1429_bs32_vocab10_repr768_msg_len4_ms...,0.919895,5e-03,7e-04,1e-03


In [6513]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,mp_similarity,mp_similarity_baseline_mean,mp_similarity_baseline_std,mp_similarity_p_value
0,coco,"[2, 3, 4]",4,coco,-,88.900 ± 0.656,0.924 ± 0.005,0.005 ± 0.000,0.001 ± 0.000,0.001 ± 0.000


# Random Length Trick

## MNIST

### Gumbel

In [6514]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

# res[gumbel_cols]

In [6515]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

,dataset,message_length,mutual_play_accuracy
0,mnist1,[3],69.1 ± 6.0
1,mnist1,[4],83.3 ± 0.8


### Gumbel - OOD

In [6516]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [6517]:
gumbel = mean_and_std(res)
# gumbel

In [6518]:
gumbel3 = gumbel.iloc[:13].reset_index(drop=True)
gumbel4 = gumbel.iloc[13:].reset_index(drop=True)

### REINFORCE - ID

In [6519]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

# res[backbone_cols]

In [6520]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

/home/mehdi_jmlkh/server2/.venv/lib/python3.12/site-packages/IPython/core/displayhook.py:281: UserWarning: Output cache limit (currently 1000 entries) hit.
Flushing oldest 200 entries.
  warn('Output cache limit (currently {sz} entries) hit.\n'


,dataset,message_length,mutual_play_accuracy
0,mnist1,[3],75.9 ± 3.6
1,mnist1,[4],85.8 ± 3.5


### REINFORCE - OOD

In [6521]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [6522]:
REINFORCE = mean_and_std(res)
# REINFORCE

In [6523]:
REINFORCE3 = REINFORCE.iloc[:13].reset_index(drop=True)
REINFORCE4 = REINFORCE.iloc[13:].reset_index(drop=True)

### VQEL - ID

In [6524]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,mnist1,cosine,[3],1e-04,-,1e-04,-,74.8,77.8,20260527_2337_bs32_vocab10_repr192_msg_len3_ms...
1,2,False,mnist1,cosine,[3],1e-04,-,1e-04,-,75.8,77.9,20260528_0025_bs32_vocab10_repr192_msg_len3_ms...
2,3,False,mnist1,cosine,[3],1e-04,-,1e-04,-,74.3,76.6,20260528_0113_bs32_vocab10_repr192_msg_len3_ms...
3,1,False,mnist1,cosine,[4],1e-04,-,1e-04,-,90.5,90.8,20251225_0626_bs32_vocab10_repr192_lr1_0.0001_...
4,2,False,mnist1,cosine,[4],1e-04,-,1e-04,-,90.1,91.9,20260527_0040_bs32_vocab10_repr192_msg_len4_ms...
5,3,False,mnist1,cosine,[4],1e-04,-,1e-04,-,91.2,90.9,20260527_0138_bs32_vocab10_repr192_msg_len4_ms...


In [6525]:
mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)

,dataset,message_length,self_play_accuracy_a,mutual_play_accuracy
0,mnist1,[3],75.0 ± 0.8,77.4 ± 0.7
1,mnist1,[4],90.6 ± 0.6,91.2 ± 0.6


### VQEL - OOD

In [6526]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [6527]:
VQ_NoTT = mean_and_std(res)
# VQ_NoTT

In [6528]:
VQ_NoTT3 = VQ_NoTT.iloc[:13].reset_index(drop=True)
VQ_NoTT4 = VQ_NoTT.iloc[13:].reset_index(drop=True)

### Batch Adaptation

In [6529]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[adapt_cols]

In [6530]:
batch_adapt = mean_and_std(res)
# batch_adapt

In [6531]:
batch_adapt3 = batch_adapt.iloc[:13].reset_index(drop=True)
batch_adapt4 = batch_adapt.iloc[13:].reset_index(drop=True)

### Dataset Adaptation

In [6532]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[adapt_cols]

In [6533]:
dataset_adapt = mean_and_std(res)
# dataset_adapt

In [6534]:
dataset_adapt3 = dataset_adapt.iloc[:13].reset_index(drop=True)
dataset_adapt4 = dataset_adapt.iloc[13:].reset_index(drop=True)

### Plot

In [6535]:
plot(
    [gumbel3, REINFORCE3, VQ_NoTT3, batch_adapt3, dataset_adapt3],
    name="rlt/mnist_without_rlt_l3",
)

In [6536]:
plot(
    [gumbel4, REINFORCE4, VQ_NoTT4, batch_adapt4, dataset_adapt4],
    name="rlt/mnist_without_rlt_l4",
)

## ImageNet

### Gumbel - ID

In [6537]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[gumbel_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase2_a,learning_rate_phase2_b,tau_0,mutual_play_accuracy,sampling_temperature,test_time_mode,path
0,1,True,imagenet,cosine,[3],-,1e-05,1e-01,87,1,-,20260528_0202_bs32_vocab10_repr2048_msg_len3_m...
1,2,True,imagenet,cosine,[3],-,1e-05,1e-01,86,1e-05,-,20260528_0226_bs32_vocab10_repr2048_msg_len3_m...
2,3,True,imagenet,cosine,[3],-,1e-05,1e-01,86.7,1e-05,-,20260528_0250_bs32_vocab10_repr2048_msg_len3_m...
3,1,True,imagenet,cosine,[4],-,1e-05,1e-01,91.8,1,-,20260527_0548_bs32_vocab10_repr2048_msg_len4_m...
4,2,True,imagenet,cosine,[4],-,1e-05,1e-01,92.4,1e-05,-,20260527_0615_bs32_vocab10_repr2048_msg_len4_m...
5,3,True,imagenet,cosine,[4],-,1e-05,1e-01,92.7,1e-05,-,20260527_0641_bs32_vocab10_repr2048_msg_len4_m...


In [6538]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

,dataset,message_length,mutual_play_accuracy
0,imagenet,[3],86.6 ± 0.5
1,imagenet,[4],92.3 ± 0.5


### Gumbel - OOD

In [6539]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [6540]:
gumbel = mean_and_std(res)
# gumbel

In [6541]:
gumbel3 = gumbel.iloc[:13].reset_index(drop=True)
gumbel4 = gumbel.iloc[13:].reset_index(drop=True)

### REINFORCE - ID

In [6542]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,True,imagenet,cosine,[3],1e-05,-,1e-05,-,0,81.1,20260528_0314_bs32_vocab10_repr2048_msg_len3_m...
1,2,True,imagenet,cosine,[3],1e-05,-,1e-05,-,0,80.4,20260528_0338_bs32_vocab10_repr2048_msg_len3_m...
2,3,True,imagenet,cosine,[3],1e-05,-,1e-05,-,0,81.3,20260528_0401_bs32_vocab10_repr2048_msg_len3_m...
3,1,True,imagenet,cosine,[4],1e-05,-,1e-05,-,0,85.4,20260527_0926_bs32_vocab10_repr2048_msg_len4_m...
4,2,True,imagenet,cosine,[4],1e-05,-,1e-05,-,0,87.9,20260527_0956_bs32_vocab10_repr2048_msg_len4_m...
5,3,True,imagenet,cosine,[4],1e-05,-,1e-05,-,0,87,20260527_1026_bs32_vocab10_repr2048_msg_len4_m...


In [6543]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

,dataset,message_length,mutual_play_accuracy
0,imagenet,[3],80.9 ± 0.5
1,imagenet,[4],86.8 ± 1.3


### REINFORCE - OOD

In [6544]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [6545]:
REINFORCE = mean_and_std(res)
# REINFORCE

In [6546]:
REINFORCE3 = REINFORCE.iloc[:13].reset_index(drop=True)
REINFORCE4 = REINFORCE.iloc[13:].reset_index(drop=True)

### VQEL - ID

In [6547]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

,seed,baseline,dataset,sim,message_length,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
0,1,False,imagenet,cosine,[3],1e-04,-,1e-04,-,83.2,82.8,20260528_0425_bs32_vocab10_repr2048_msg_len3_m...
1,2,False,imagenet,cosine,[3],1e-04,-,1e-04,-,83.5,84.3,20260528_0440_bs32_vocab10_repr2048_msg_len3_m...
2,3,False,imagenet,cosine,[3],1e-04,-,1e-04,-,84.6,84.7,20260528_0454_bs32_vocab10_repr2048_msg_len3_m...
3,1,False,imagenet,cosine,[4],1e-04,-,1e-04,-,92.3,92.1,20260527_0236_bs32_vocab10_repr2048_msg_len4_m...
4,2,False,imagenet,cosine,[4],1e-04,-,1e-04,-,92.7,91.1,20260527_0252_bs32_vocab10_repr2048_msg_len4_m...
5,3,False,imagenet,cosine,[4],1e-04,-,1e-04,-,91.7,92.4,20260527_0308_bs32_vocab10_repr2048_msg_len4_m...


In [6548]:
mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)

,dataset,message_length,self_play_accuracy_a,mutual_play_accuracy
0,imagenet,[3],83.8 ± 0.7,83.9 ± 1.0
1,imagenet,[4],92.2 ± 0.5,91.9 ± 0.7


### VQEL - OOD

In [6549]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [6550]:
VQ_NoTT = mean_and_std(res)
VQ_NoTT

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,[3],3,imagenet_same_class,-,75.2 ± 0.8,37.3 ± 1.6
1,imagenet,[3],4,imagenet_same_class,-,11.2 ± 3.0,9.3 ± 0.9
2,imagenet,[3],5,imagenet_same_class,-,4.4 ± 1.8,5.9 ± 1.2
3,imagenet,[3],6,imagenet_same_class,-,17.3 ± 6.3,7.3 ± 1.5
4,imagenet,[3],7,imagenet_same_class,-,1.6 ± 1.4,3.0 ± 1.7
5,imagenet,[3],8,imagenet_same_class,-,6.5 ± 1.2,4.9 ± 1.7
6,imagenet,[3],9,imagenet_same_class,-,4.7 ± 3.5,3.2 ± 1.6
7,imagenet,[3],10,imagenet_same_class,-,3.6 ± 1.5,4.6 ± 1.7
8,imagenet,[3],11,imagenet_same_class,-,9.9 ± 2.5,5.7 ± 0.3
9,imagenet,[3],12,imagenet_same_class,-,1.7 ± 1.4,2.4 ± 0.5


In [6551]:
VQ_NoTT3 = VQ_NoTT.iloc[:13].reset_index(drop=True)
VQ_NoTT4 = VQ_NoTT.iloc[13:].reset_index(drop=True)

### Batch Adaptation

In [6552]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

res[adapt_cols]

,seed,baseline,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,number_of_candidates,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,1,False,imagenet,[3],3,imagenet_same_class,batch_adaptation,32,1e-04,200,50.1,43.7,20260528_1309_bs32_vocab10_repr2048_msg_len3_m...
1,2,False,imagenet,[3],3,imagenet_same_class,batch_adaptation,32,1e-04,200,48.2,43.2,20260528_1312_bs32_vocab10_repr2048_msg_len3_m...
2,3,False,imagenet,[3],3,imagenet_same_class,batch_adaptation,32,1e-04,200,47.5,43.2,20260528_1315_bs32_vocab10_repr2048_msg_len3_m...
3,1,False,imagenet,[3],4,imagenet_same_class,batch_adaptation,32,1e-04,200,63.3,45.4,20260528_1318_bs32_vocab10_repr2048_msg_len4_m...
4,2,False,imagenet,[3],4,imagenet_same_class,batch_adaptation,32,1e-04,200,64.1,47.4,20260528_1321_bs32_vocab10_repr2048_msg_len4_m...
5,3,False,imagenet,[3],4,imagenet_same_class,batch_adaptation,32,1e-04,200,61,44.2,20260528_1325_bs32_vocab10_repr2048_msg_len4_m...
6,1,False,imagenet,[3],5,imagenet_same_class,batch_adaptation,32,1e-04,200,70.7,44.4,20260528_1328_bs32_vocab10_repr2048_msg_len5_m...
7,2,False,imagenet,[3],5,imagenet_same_class,batch_adaptation,32,1e-04,200,70.3,46.3,20260528_1332_bs32_vocab10_repr2048_msg_len5_m...
8,3,False,imagenet,[3],5,imagenet_same_class,batch_adaptation,32,1e-04,200,67.9,46.6,20260528_1336_bs32_vocab10_repr2048_msg_len5_m...
9,1,False,imagenet,[3],6,imagenet_same_class,batch_adaptation,32,1e-04,200,70.7,47.8,20260528_1340_bs32_vocab10_repr2048_msg_len6_m...


In [6553]:
batch_adapt = mean_and_std(res)
# batch_adapt

In [6554]:
batch_adapt3 = batch_adapt.iloc[:13].reset_index(drop=True)
batch_adapt4 = batch_adapt.iloc[13:].reset_index(drop=True)

### Dataset Adaptation

In [6555]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[adapt_cols]

In [6556]:
dataset_adapt = mean_and_std(res)
# dataset_adapt

In [6557]:
dataset_adapt3 = dataset_adapt.iloc[:13].reset_index(drop=True)
dataset_adapt4 = dataset_adapt.iloc[13:].reset_index(drop=True)

### Plot

In [6558]:
plot(
    [gumbel3, REINFORCE3, VQ_NoTT3, batch_adapt3, dataset_adapt3],
    name="rlt/imagenet_without_rlt_l3",
)

In [6559]:
plot(
    [gumbel4, REINFORCE4, VQ_NoTT4, batch_adapt4, dataset_adapt4],
    name="rlt/imagenet_without_rlt_l4",
)

# Full sender Adaptation

## MNIST

### Batch Adaptation

In [6560]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "full_sender_batch_adaptation",
        "dataset_tt": "mnist2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6561]:
final = extract_maxes(res)
# final[adapt_cols]

In [6562]:
batch_adapt = mean_and_std(final)
# batch_adapt

### Dataset Adaptation

In [6563]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "full_sender_dataset_adaptation",
        "dataset_tt": "mnist2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6564]:
dataset_adapt = mean_and_std(res)
# dataset_adapt

### Plot

In [6565]:
# plot([mnist_gumbel, mnist_REINFORCE, mnist_VQ_NoTT, batch_adapt, mnist_batch_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/mnist_batch")

In [6566]:
# plot([mnist_gumbel, mnist_REINFORCE, mnist_VQ_NoTT, dataset_adapt, mnist_dataset_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/mnist_dataset")

## ImageNet

### Batch Adaptation

In [6567]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "full_sender_batch_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6568]:
final = extract_maxes(res)
# final[adapt_cols]

In [6569]:
batch_adapt = mean_and_std(final)
# batch_adapt

### Dataset Adaptation

In [6570]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "full_sender_dataset_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [6571]:
dataset_adapt = mean_and_std(res)
dataset_adapt

,dataset,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy
0,imagenet,"[2, 3, 4]",4,imagenet_same_class,full_sender_dataset_adaptation,85.4 ± 1.0,32.6 ± 1.9
1,imagenet,"[2, 3, 4]",5,imagenet_same_class,full_sender_dataset_adaptation,91.8 ± 0.3,36.5 ± 3.2
2,imagenet,"[2, 3, 4]",6,imagenet_same_class,full_sender_dataset_adaptation,95.1 ± 0.5,33.8 ± 1.9
3,imagenet,"[2, 3, 4]",7,imagenet_same_class,full_sender_dataset_adaptation,97.3 ± 0.8,34.4 ± 0.5
4,imagenet,"[2, 3, 4]",8,imagenet_same_class,full_sender_dataset_adaptation,97.2 ± 0.6,31.6 ± 2.0
5,imagenet,"[2, 3, 4]",9,imagenet_same_class,full_sender_dataset_adaptation,97.9 ± 0.1,32.4 ± 1.3
6,imagenet,"[2, 3, 4]",10,imagenet_same_class,full_sender_dataset_adaptation,98.5 ± 0.4,32.0 ± 1.8
7,imagenet,"[2, 3, 4]",11,imagenet_same_class,full_sender_dataset_adaptation,98.7 ± 0.4,29.2 ± 1.9
8,imagenet,"[2, 3, 4]",12,imagenet_same_class,full_sender_dataset_adaptation,99.0 ± 0.6,29.4 ± 1.2
9,imagenet,"[2, 3, 4]",13,imagenet_same_class,full_sender_dataset_adaptation,99.4 ± 0.4,27.8 ± 2.0


### Plot

In [6572]:
# plot([imagenet_gumbel, imagenet_REINFORCE, imagenet_VQ_NoTT, batch_adapt, imagenet_batch_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/imagenet_batch")

In [6573]:
# plot([imagenet_gumbel, imagenet_REINFORCE, imagenet_VQ_NoTT, dataset_adapt, imagenet_dataset_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/imagenet_dataset")